# 1. Install AG2 + Materials Project dependencies

In [1]:
!pip install -U "ag2[openai]" autogen -q

import sys
!{sys.executable} -m pip install pymatgen mp-api numpy cython ipywidgets jupyterlab_widgets -q

print("✓ AG2/autogen + Pymatgen + MP-API installed.")

✓ AG2/autogen + Pymatgen + MP-API installed.


# 2. Imports + LLM Configuration


In [2]:
import subprocess
import shutil

if shutil.which("tectonic") is None:
    subprocess.run(
        ["conda", "install", "-c", "conda-forge", "tectonic", "-y", "--quiet"],
        check=False
    )

if shutil.which("tectonic") is not None:
    print("\u2713 Tectonic (LaTeX compiler) ready.")
else:
    print("\u26a0\ufe0f  Tectonic not found. Run manually: conda install -c conda-forge tectonic")

import os
import json
import random
import base64
import re
from datetime import datetime, timedelta, timezone
from typing import Annotated, Any, Literal, Union, List, Optional, Tuple
from enum import Enum
from pathlib import Path

from openai import OpenAI

from autogen import ConversableAgent
from autogen.agentchat import ReplyResult
from autogen.agentchat.group import (
    ContextVariables,
    AgentTarget, AgentNameTarget, StayTarget,
    OnCondition, StringLLMCondition,
    OnContextCondition, ExpressionContextCondition, ContextExpression,
    RevertToUserTarget, TerminateTarget
)
from autogen.agentchat.groupchat import GroupChat, GroupChatManager
from autogen.tools import tool

from mp_api.client import MPRester

from autogen.coding.local_commandline_code_executor import LocalCommandLineCodeExecutor
from autogen.coding.base import CodeBlock
from pydantic import BaseModel, Field

from autogen.agentchat.group.patterns import DefaultPattern
from autogen.agentchat import initiate_group_chat

from paths import PATHS


# --- LLM CONFIGURATION


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

llm_config = {
    "model": "gpt-4o",
    "api_key": os.environ["OPENAI_API_KEY"],
}


def llm_completion(
    *,
    client: OpenAI,
    model: str,
    input: list,
    text_format=None,
    max_tokens: int = 2000,
    temperature: float = 0.2,
):
    if not isinstance(model, str) or not model.strip():
        raise ValueError("Model name must be a non-empty string.")

    model_name = model.strip()
    reasoning_model = model_name.startswith("o")

    if text_format is not None:
        kwargs = dict(
            model=model_name,
            input=input,
            text_format=text_format,
            max_output_tokens=max_tokens,
        )

        if not reasoning_model:
            kwargs["temperature"] = temperature

        resp = client.responses.parse(**kwargs)
        return resp.output_parsed

    kwargs = dict(
        model=model_name,
        input=input,
        max_output_tokens=max_tokens,
    )

    if not reasoning_model:
        kwargs["temperature"] = temperature

    resp = client.responses.create(**kwargs)

    return getattr(resp, "output_text", None)


# --- PROJECT FOLDER


PROJECT_FOLDER = os.path.abspath("ag2_project")
os.makedirs(PROJECT_FOLDER, exist_ok=True)
os.environ["PROJECT_FOLDER"] = PROJECT_FOLDER

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

PLOTS_V1_DIR = PATHS["plots_v1"]
PLOTS_V2_DIR = PATHS["plots_v2"]

print("✓ Imports loaded and LLM configured.")

✓ Tectonic (LaTeX compiler) ready.
✓ Imports loaded and LLM configured.


# 3. Data Contracts — (Plot Artifacts + Visual Validation, Writer)

In [3]:
# --- Plot Contracts ---

class PlotType(str, Enum):
    scatter_2d = "scatter_2d"
    histogram = "histogram"
    bar = "bar"
    heatmap_2d = "heatmap_2d"
    scatter_3d = "scatter_3d"
    pie = "pie"


class PlotAxes(BaseModel):
    type: Literal["2d", "3d", "hist", "heatmap", "categorical", "pie"] = Field(...)
    x: Optional[str] = None
    y: Optional[str] = None
    z: Optional[str] = None
    intensity: Optional[str] = None


class PlotCandidate(BaseModel):
    plot_id: str = Field(..., min_length=1)
    title: str = Field(..., min_length=1)
    plot_type: PlotType
    data_requirements: List[str]
    axes: PlotAxes

    def validate_structure(self):
        if self.plot_type == PlotType.scatter_2d:
            if self.axes.type != "2d":
                raise ValueError("scatter_2d requires axes.type='2d'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("scatter_2d requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("scatter_2d cannot include z or intensity")

        if self.plot_type == PlotType.histogram:
            if self.axes.type != "hist":
                raise ValueError("histogram requires axes.type='hist'")
            if not self.axes.x:
                raise ValueError("histogram requires axes.x")
            if self.axes.y or self.axes.z or self.axes.intensity:
                raise ValueError("histogram only supports axes.x")

        if self.plot_type == PlotType.bar:
            if self.axes.type != "categorical":
                raise ValueError("bar requires axes.type='categorical'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("bar requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("bar cannot include z or intensity")

        if self.plot_type == PlotType.heatmap_2d:
            if self.axes.type != "heatmap":
                raise ValueError("heatmap_2d requires axes.type='heatmap'")
            if not self.axes.x or not self.axes.y or not self.axes.intensity:
                raise ValueError("heatmap_2d requires axes.x, axes.y and axes.intensity")
            if self.axes.z:
                raise ValueError("heatmap_2d cannot include z")

        if self.plot_type == PlotType.scatter_3d:
            if self.axes.type != "3d":
                raise ValueError("scatter_3d requires axes.type='3d'")
            if not self.axes.x or not self.axes.y or not self.axes.z:
                raise ValueError("scatter_3d requires axes.x, axes.y and axes.z")
            if self.axes.intensity:
                raise ValueError("scatter_3d cannot include intensity")

        if self.plot_type == PlotType.pie:
            if self.axes.type != "pie":
                raise ValueError("pie requires axes.type='pie'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("pie requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("pie cannot include z or intensity")


# --- Plot Artifacts ---

class PlotArtifactMeta(BaseModel):
    plot_id: str = Field(..., min_length=1)
    name: str = Field(..., min_length=1)
    path: str = Field(..., min_length=1)
    description: str = Field(..., min_length=1)


def validate_plot_artifact_meta(item: dict) -> dict:
    meta = PlotArtifactMeta(**item)

    if not meta.name.startswith(f"{meta.plot_id}_"):
        raise ValueError("name must start with plot_id + '_'")

    if os.path.isabs(meta.path):
        raise ValueError("path must be relative")

    if not (meta.path.startswith("plots/") or meta.path.startswith("plots_v2/")):
        raise ValueError("path must start with 'plots/' or 'plots_v2/'")

    return meta.model_dump()


# --- Visual Validation ---

class PlotValidationResult(BaseModel):
    summary: str
    issues: List[str]
    improvements: List[str]
    severity: Literal["low", "medium", "high"]
    pass_fail: bool


def evaluate_plot(
    plot_name: str,
    plot_description: str,
    model: str,
    plot_path: str,
) -> PlotValidationResult:
    import base64

    if not isinstance(plot_name, str) or not plot_name.strip():
        raise ValueError("plot_name must be a non-empty string.")

    if not isinstance(plot_description, str) or not plot_description.strip():
        raise ValueError("plot_description must be a non-empty string.")

    if not isinstance(model, str) or not model.strip():
        raise ValueError("model must be a non-empty string.")

    if not isinstance(plot_path, str) or not plot_path.strip():
        raise ValueError("plot_path must be a non-empty string.")

    rel_path = plot_path.strip().replace("\\", "/")
    if os.path.isabs(rel_path) or ".." in rel_path.split("/"):
        raise ValueError("plot_path must be a safe relative path.")

    input_image = os.path.join(PROJECT_FOLDER, rel_path)

    if not os.path.exists(input_image):
        raise RuntimeError(f"Plot image not found at '{input_image}'")

    try:
        with open(input_image, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"Failed to read plot image at '{input_image}': {e}")

    prompt = (
        "You are a strict plot reviewer.\n"
        "Provide qualitative visual feedback only.\n"
        "Focus on:\n"
        "- readability\n"
        "- axis labels\n"
        "- units\n"
        "- scaling\n"
        "- visual clarity\n"
        "Do NOT perform exact numeric reading.\n\n"
        f"Plot task description:\n{plot_description.strip()}"
    )

    parsed = llm_completion(
        client=client,
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{base64_image}",
                    },
                    {
                        "type": "input_text",
                        "text": prompt,
                    },
                ],
            }
        ],
        text_format=PlotValidationResult,
        max_tokens=10000,
    )

    if parsed is None:
        raise RuntimeError("LLM returned None during plot evaluation.")

    if not isinstance(parsed, PlotValidationResult):
        raise RuntimeError("LLM returned an invalid PlotValidationResult object.")

    return parsed


# --- Writer Contracts ---

class LatexSections(BaseModel):
    latex_sections: str = Field(..., min_length=1)


def validate_latex_sections(item: dict) -> dict:
    obj = LatexSections(**item)
    s = obj.latex_sections.strip()

    forbidden = [
        "\\documentclass",
        "\\usepackage",
        "\\begin{document}",
        "\\end{document}",
    ]
    for token in forbidden:
        if token in s:
            raise ValueError("latex_sections must not include LaTeX preamble or document environment.")

    if "\\begin{abstract}" not in s or "\\end{abstract}" not in s:
        raise ValueError("latex_sections must include an abstract environment.")

    return obj.model_dump()

# 4. User query + Context Variables

In [4]:
from autogen.agentchat.group import ContextVariables

context_variables = ContextVariables(
    data={
        "task_started": False,
        "query": None,

        "main_task_analysis": None,
        "main_task_improvements": None,
        "main_task_improved": None,

        "plan": None,
        "review_status": None,
        "review_notes": None,

        "search_criteria": None,
        "fields": None,
        "sample_number": None,
        "mp_results": None,

        "final_conclusion": None,
        "analysis_summary": None,
        "materials_data_path": None,

        "plot_candidates": [],
        "plot_selected": [],
        "plot_selection_notes": None,
        "plots_v1": [],
        "plot_reviews": {},
        "plots_v2": [],
        "plots_output_dir_v1": "plots",
        "plots_output_dir_v2": "plots_v2",
        "plots_path_v1": PLOTS_V1_DIR,
        "plots_path_v2": PLOTS_V2_DIR,

        "plot_suggest_status": "idle",
        "plot_select_status": "idle",
        "plot_code_status": "idle",
        "plot_validate_status": "idle",

        "image_artifacts_queue": [],
        "image_reviews": [],

        "last_executed_code": None,
        "last_execution_output": None,
        "last_execution_exit_code": None,
        "last_executed_file": None,

        "latex_sections": None,
        "report_sections_path": None,
        "report_tex_path": None,
        "report_tex_path_abs": None,
        "report_pdf_path": None,
        "latex_status": None,
        "latex_engine": None,
        "latex_passes": None,
        "latex_compile_output": None,
        "latex_log_path": None,
        "latex_log_tail": None,
        "latex_compile_error": None,

        "next_agent": None,
    }
)

context_variables["agent_registry"] = [
    "AgentTaskImprover",
    "AgentPlanner",
    "AgentPlanReviewer",
    "AgentMaterialsRetriever",
    "AgentAnalyzer",
    "AgentPlotSuggester",
    "AgentPlotSelector",
    "AgentCoder",
    "AgentPlotValidator",
    "AgentLatexWriter",
    "AgentLatexCompiler",
    "Human",
]

print("✓ ContextVariables initialized.")

✓ ContextVariables initialized.


# 5. LaTeX Template

In [5]:
LATEX_TEMPLATE = r"""
\documentclass{article}

\usepackage{graphicx}
\usepackage{float}
\usepackage{geometry}
\usepackage{caption}
\usepackage{booktabs}
\usepackage{hyperref}

\geometry{margin=1in}
\graphicspath{{../../plots/}{../../plots_v2/}}

\title{Automated Materials Screening Report}
\author{AG2 Multi-Agent System}
\date{\today}

\begin{document}
\maketitle

{section_content}

\end{document}
"""

# 6. TOOLS 

In [11]:
# Tool — task_improver

def task_improver_tool(
    analysis: Annotated[str, "Short analysis of the task consistency, missing info, redundancy."],
    improvements: Annotated[str, "Concrete improvements to make the task complete and non-redundant."],
    improved_task: Annotated[str, "Improved version of the main task."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentTaskImprover"
    next_agent = "AgentPlanner"

    if context_variables.get("task_improver_status") == "completed":
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message="task improver stage already completed",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    context_variables["task_improver_status"] = "running"

    def _fail(message: str) -> ReplyResult:
        context_variables["task_improver_status"] = "failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in task_improver_tool: {message}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        if not isinstance(analysis, str) or not analysis.strip():
            raise ValueError("analysis must be a non-empty string.")
        if not isinstance(improvements, str) or not improvements.strip():
            raise ValueError("improvements must be a non-empty string.")
        if not isinstance(improved_task, str) or not improved_task.strip():
            raise ValueError("improved_task must be a non-empty string.")

        original_query = context_variables.get("query", "")
        if not isinstance(original_query, str) or not original_query.strip():
            raise ValueError("context_variables['query'] must be a non-empty string.")

    except ValueError as e:
        return _fail(str(e))

    analysis_s = analysis.strip()
    improvements_s = improvements.strip()
    improved_task_s = improved_task.strip()
    original_query_s = original_query.strip()

    lowered_task = improved_task_s.lower()
    lowered_query = original_query_s.lower()

    def _normalize(text: str) -> str:
        text = text.replace("g/cm³", "g/cm3")
        text = text.replace("g/cm^3", "g/cm3")
        text = re.sub(r"\s+", " ", text)
        return text.strip().lower()

    def _extract_numeric_ranges(text: str):
        pattern = r"([a-zA-Z_]+)\s+(?:between|from)\s+([0-9]*\.?[0-9]+)\s+(?:and|to)\s+([0-9]*\.?[0-9]+)"
        out = {}
        for field, lo, hi in re.findall(pattern, _normalize(text)):
            out[field] = (lo, hi)
        return out

    def _extract_excluded_elements(text: str):
        m = re.search(
            r"excluding toxic elements such as\s+([A-Z][a-z]?(?:\s*,\s*[A-Z][a-z]?)*(?:\s*,?\s*and\s+[A-Z][a-z]?)?)",
            text,
        )
        if not m:
            return []
        raw = m.group(1)
        return re.findall(r"\b[A-Z][a-z]?\b", raw)

    forbidden_added_requirements = [
        "png format",
        "pdf format",
        "best candidate",
        "top candidate",
        "rank the candidates",
        "ranking the candidates",
        "rank candidates",
        "multiple viable candidates",
        "comparison plot",
        "scatter plot",
        "histogram",
        "heatmap",
        "bar chart",
        "3d plot",
        "report in pdf",
        "scientific report",
        "or their compounds",
        "their compounds",
        "indirect compounds",
        "safety standards",
        "environmental compliance",
        "real-world implications",
        "practical requirements",
    ]

    for token in forbidden_added_requirements:
        if token in lowered_task and token not in lowered_query:
            return _fail(
                "improved_task introduces extra requirements "
                f"that change the original scope ('{token}')."
            )

    query_mentions_plotting = any(
        token in lowered_query
        for token in ["plot", "plots", "plotting", "graph", "graphs", "chart", "charts", "visualization"]
    )

    task_mentions_over_specific_plotting = any(
        token in lowered_task
        for token in [
            "comparison plot",
            "scatter plot",
            "histogram",
            "heatmap",
            "bar chart",
            "3d plot",
            "plot relevant material properties",
            "plot of",
            "plotting step",
            "data visualization step",
        ]
    )

    if query_mentions_plotting and task_mentions_over_specific_plotting:
        generic_plot_request = not any(
            token in lowered_query
            for token in [
                "comparison plot",
                "scatter plot",
                "histogram",
                "heatmap",
                "bar chart",
                "3d plot",
            ]
        )
        if generic_plot_request:
            return _fail(
                "improved_task makes plotting more specific than the original query."
            )

    for phrase in ["stable", "include plotting in the workflow"]:
        if phrase in _normalize(original_query_s) and phrase not in _normalize(improved_task_s):
            return _fail(
                f"improved_task removes or weakens an explicit user requirement ('{phrase}')."
            )

    query_ranges = _extract_numeric_ranges(original_query_s)
    task_ranges = _extract_numeric_ranges(improved_task_s)

    for field, values in query_ranges.items():
        if field not in task_ranges:
            return _fail(
                f"improved_task removes the numeric range for '{field}'."
            )
        if task_ranges[field] != values:
            return _fail(
                f"improved_task changes the numeric range for '{field}' from {values} to {task_ranges[field]}."
            )

    query_excluded = _extract_excluded_elements(original_query_s)
    task_excluded = _extract_excluded_elements(improved_task_s)

    if query_excluded:
        if task_excluded != query_excluded:
            return _fail(
                "improved_task changes the explicit excluded elements or their order."
            )

    if "excluding toxic elements such as" in _normalize(original_query_s):
        if "excluding toxic elements such as" not in _normalize(improved_task_s):
            return _fail(
                "improved_task weakens or alters the original exclusion wording."
            )

    suspicious_additions = [
        "compliance",
        "environmental safety",
        "environmental harm",
        "insulation",
        "outdoor",
        "protects infrastructure",
        "high-voltage areas",
    ]

    normalized_query = _normalize(original_query_s)
    normalized_task = _normalize(improved_task_s)

    for token in suspicious_additions:
        if token in normalized_task and token not in normalized_query:
            return _fail(
                f"improved_task adds inferred context that was not explicitly requested ('{token}')."
            )

    context_variables["task_started"] = True
    context_variables["main_task_analysis"] = analysis_s
    context_variables["main_task_improvements"] = improvements_s
    context_variables["main_task_improved"] = improved_task_s
    context_variables["task_improver_status"] = "completed"
    context_variables["next_agent"] = next_agent

    message = {
        "main_task_analysis": analysis_s,
        "main_task_improvements": improvements_s,
        "main_task_improved": improved_task_s,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — planner

def planner_tool(
    plan: Annotated[dict | None, "Structured plan dict generated by the Planner agent."],
    plan_description: Annotated[str | None, "Short description of the plan."],
    success_criteria: Annotated[List[str] | None, "Concrete criteria to consider the run successful."],
    plan_score: Annotated[float | None, "Planner self-score for the plan quality (0-10)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlanner"
    main_task_improved = context_variables.get("main_task_improved", None)

    if context_variables.get("planner_status") == "completed":
        existing_plan = context_variables.get("plan", None)
        existing_plan_description = context_variables.get("plan_description", None)
        existing_success_criteria = context_variables.get("success_criteria", None)
        existing_plan_score = context_variables.get("plan_score", None)

        next_agent_existing = "AgentPlanReviewer"
        if isinstance(existing_plan, dict):
            routing_existing = existing_plan.get("routing", {})
            if isinstance(routing_existing, dict):
                after_planner_existing = routing_existing.get("after_planner", None)
                if isinstance(after_planner_existing, str) and after_planner_existing.strip():
                    next_agent_existing = after_planner_existing.strip()

        context_variables["next_agent"] = next_agent_existing

        message = {
            "plan_description": existing_plan_description,
            "success_criteria": existing_success_criteria,
            "plan_score": existing_plan_score,
            "plan": existing_plan,
        }

        return ReplyResult(
            message=json.dumps(message, indent=2, ensure_ascii=False, default=str),
            target=AgentNameTarget(next_agent_existing),
            context_variables=context_variables,
        )

    context_variables["planner_status"] = "running"

    try:
        if not isinstance(main_task_improved, str) or not main_task_improved.strip():
            raise ValueError("context_variables['main_task_improved'] must be a non-empty string.")
        if not isinstance(plan, dict) or not plan:
            raise ValueError("plan must be a non-empty dict.")
        if not isinstance(plan_description, str) or not plan_description.strip():
            raise ValueError("plan_description must be a non-empty string.")
        if not isinstance(success_criteria, list) or not success_criteria or not all(
            isinstance(x, str) and x.strip() for x in success_criteria
        ):
            raise ValueError("success_criteria must be a non-empty list of non-empty strings.")
        if not isinstance(plan_score, (int, float)):
            raise ValueError("plan_score must be a number.")
        plan_score_f = float(plan_score)
        if plan_score_f < 0 or plan_score_f > 10:
            raise ValueError("plan_score must be between 0 and 10.")
    except ValueError as e:
        context_variables["planner_status"] = "failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in planner_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    main_task_s = main_task_improved.strip()

    try:
        if not isinstance(plan.get("schema_version"), str) or not plan["schema_version"].strip():
            raise ValueError("plan.schema_version must be a non-empty string.")

        if not isinstance(plan.get("main_task_improved"), str) or not plan["main_task_improved"].strip():
            raise ValueError("plan.main_task_improved must be a non-empty string.")
        if plan["main_task_improved"].strip() != main_task_s:
            raise ValueError("plan.main_task_improved must match context_variables['main_task_improved'].")

        if not isinstance(plan.get("fireground_signals"), dict):
            raise ValueError("plan.fireground_signals must be a dict.")
        if not isinstance(plan.get("material_roles"), dict):
            raise ValueError("plan.material_roles must be a dict.")

        routing = plan.get("routing")
        if not isinstance(routing, dict):
            raise ValueError("plan.routing must be a dict.")

        required_routing = {
            "after_planner": "AgentPlanReviewer",
            "on_approve": "AgentMaterialsRetriever",
            "on_revise": "AgentTaskImprover",
            "on_handoff": "Human",
            "after_plot_suggest": "AgentPlotSelector",
            "after_plot_select": "AgentCoder",
            "after_code": "AgentPlotValidator",
            "after_plot_validate": "AgentLatexWriter",
            "on_plot_validation_failed": "AgentCoder",
            "after_latex_write": "AgentLatexCompiler",
            "after_latex_compile": "Human",
            "on_latex_compile_failed": "AgentLatexWriter",
        }

        for k, expected_value in required_routing.items():
            if not isinstance(routing.get(k), str) or not routing[k].strip():
                raise ValueError(f"plan.routing.{k} must be a non-empty string.")
            if routing[k].strip() != expected_value:
                raise ValueError(f"plan.routing.{k} must be '{expected_value}'.")

        steps = plan.get("steps")
        if not isinstance(steps, list) or len(steps) == 0:
            raise ValueError("plan.steps must be a non-empty list.")
        if not all(isinstance(s, dict) for s in steps):
            raise ValueError("plan.steps must contain dict items only.")
        if not all(isinstance(s.get("id"), str) and s["id"].strip() for s in steps):
            raise ValueError("Each step must have a non-empty string 'id'.")
        if not all(isinstance(s.get("agent"), str) and s["agent"].strip() for s in steps):
            raise ValueError("Each step must have a non-empty string 'agent'.")

        allowed_agents = {
            "AgentTaskImprover",
            "AgentPlanner",
            "AgentPlanReviewer",
            "AgentMaterialsRetriever",
            "AgentAnalyzer",
            "AgentPlotSuggester",
            "AgentPlotSelector",
            "AgentPlotValidator",
            "AgentCoder",
            "AgentLatexWriter",
            "AgentLatexCompiler",
            "Human",
        }

        for s in steps:
            if s["agent"] not in allowed_agents:
                raise ValueError(f"Step '{s.get('id')}' has invalid agent '{s['agent']}'.")

        required_step_agents = {
            "retrieve": "AgentMaterialsRetriever",
            "analyze": "AgentAnalyzer",
            "plot_suggest": "AgentPlotSuggester",
            "plot_select": "AgentPlotSelector",
            "plot_generate_v1": "AgentCoder",
            "plot_validate": "AgentPlotValidator",
            "plot_generate_v2": "AgentCoder",
            "latex_write": "AgentLatexWriter",
            "latex_compile": "AgentLatexCompiler",
        }

        steps_by_id = {}
        for s in steps:
            step_id = s["id"].strip()
            if step_id in steps_by_id:
                raise ValueError(f"Duplicate step id '{step_id}' in plan.steps.")
            steps_by_id[step_id] = s

        for required_id, required_agent in required_step_agents.items():
            if required_id not in steps_by_id:
                raise ValueError(f"Missing required step '{required_id}'.")
            if steps_by_id[required_id]["agent"] != required_agent:
                raise ValueError(
                    f"Step '{required_id}' must use agent '{required_agent}'."
                )

        retrieval = plan.get("retrieval")
        if not isinstance(retrieval, dict):
            raise ValueError("plan.retrieval must be a dict.")
        if not isinstance(retrieval.get("search_criteria"), dict):
            raise ValueError("plan.retrieval.search_criteria must be a dict.")

        search_criteria = retrieval.get("search_criteria")

        allowed_search_keys = {
            "band_gap",
            "energy_above_hull",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
            "elements",
            "chemsys",
            "exclude_elements",
        }

        found_keys = set(search_criteria.keys())
        invalid_keys = sorted(found_keys - allowed_search_keys)
        retrieval_errors = []

        if invalid_keys:
            retrieval_errors.append(
                f"plan.retrieval.search_criteria contains invalid keys: {invalid_keys}"
            )

        if "excluded_elements" in search_criteria:
            retrieval_errors.append(
                "plan.retrieval.search_criteria must use 'exclude_elements', not 'excluded_elements'."
            )

        numeric_range_keys = {
            "energy_above_hull",
            "band_gap",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
        }

        normalized_search_criteria = dict(search_criteria)

        for key in numeric_range_keys:
            if key in search_criteria:
                value = search_criteria[key]
                if not (
                    isinstance(value, (list, tuple))
                    and len(value) == 2
                    and all(isinstance(x, (int, float)) for x in value)
                ):
                    retrieval_errors.append(
                        f"plan.retrieval.search_criteria.{key} must be a list/tuple(min, max) of numbers."
                    )
                else:
                    min_v, max_v = value
                    if min_v > max_v:
                        retrieval_errors.append(
                            f"plan.retrieval.search_criteria.{key} must satisfy min <= max."
                        )
                    else:
                        normalized_search_criteria[key] = [float(min_v), float(max_v)]

        if "exclude_elements" in search_criteria:
            exclude_elements = search_criteria["exclude_elements"]
            if not (
                isinstance(exclude_elements, list)
                and len(exclude_elements) > 0
                and all(isinstance(x, str) and x.strip() for x in exclude_elements)
            ):
                retrieval_errors.append(
                    "plan.retrieval.search_criteria.exclude_elements must be a non-empty list of non-empty strings."
                )
            else:
                normalized_search_criteria["exclude_elements"] = [x.strip() for x in exclude_elements]

        if retrieval_errors:
            raise ValueError(" | ".join(retrieval_errors))

        plan["retrieval"]["search_criteria"] = normalized_search_criteria

        if not isinstance(retrieval.get("fields"), list) or not retrieval["fields"] or not all(
            isinstance(x, str) and x.strip() for x in retrieval["fields"]
        ):
            raise ValueError("plan.retrieval.fields must be a non-empty list of non-empty strings.")

        allowed_fields = {
            "material_id",
            "formula_pretty",
            "band_gap",
            "energy_above_hull",
            "density",
            "volume",
            "symmetry",
            "nsites",
            "elements",
            "chemsys",
            "is_stable",
            "bulk_modulus",
            "shear_modulus",
        }

        invalid_fields = [x for x in retrieval["fields"] if x not in allowed_fields]
        if invalid_fields:
            raise ValueError(f"plan.retrieval.fields contains invalid values: {invalid_fields}")

        if not isinstance(retrieval.get("sample_number"), int) or retrieval["sample_number"] <= 0:
            raise ValueError("plan.retrieval.sample_number must be a positive int.")

        plotting = plan.get("plotting", None)
        if not isinstance(plotting, dict):
            raise ValueError("plan.plotting must be a dict.")
        if plotting.get("enabled") is not True:
            raise ValueError("plan.plotting.enabled must be True.")
        n_candidates = plotting.get("n_candidates", None)
        n_selected = plotting.get("n_selected", None)
        output_dir = plotting.get("output_dir", None)
        if not isinstance(n_candidates, int) or n_candidates != 10:
            raise ValueError("plan.plotting.n_candidates must be int == 10.")
        if not isinstance(n_selected, int) or n_selected != 5:
            raise ValueError("plan.plotting.n_selected must be int == 5.")
        if not isinstance(output_dir, str) or output_dir.strip() != "plots":
            raise ValueError('plan.plotting.output_dir must be "plots".')

        plot_intents = plotting.get("plot_intents", None)
        if plot_intents is None:
            raise ValueError("plan.plotting.plot_intents is required when plotting.enabled is True.")
        if not isinstance(plot_intents, list):
            raise ValueError("plan.plotting.plot_intents must be a list[str].")
        if len(plot_intents) == 0:
            raise ValueError("plan.plotting.plot_intents must be non-empty when plotting.enabled is True.")
        for i, intent in enumerate(plot_intents):
            if not isinstance(intent, str):
                raise ValueError(f"plan.plotting.plot_intents[{i}] must be a string.")
            if not intent.strip():
                raise ValueError(f"plan.plotting.plot_intents[{i}] must be a non-empty string.")

        retrieved_fields = {x.strip() for x in retrieval["fields"] if isinstance(x, str) and x.strip()}
        allowed_direct_plot_fields = set(retrieved_fields)
        allowed_derived_plot_fields = {"stability_bins", "band_gap_bins"}

        vague_tokens = {
            "relevant properties",
            "material suitability",
            "suitability",
            "stability metrics",
            "material performance",
            "performance metrics",
            "relevant material properties",
            "plot relevant properties",
            "plot material suitability",
            "plot stability metrics",
            "plot material performance",
        }

        preferred_intents = {
            "scatter energy_above_hull vs band_gap",
            "scatter density vs band_gap",
            "histogram energy_above_hull",
            "histogram density",
            "bar stability_bins",
            "bar band_gap_bins",
        }

        blocked_direct_intents = {
            "bar band_gap_bins",
        }

        for i, intent in enumerate(plot_intents):
            intent_s = intent.strip()
            intent_l = intent_s.lower()

            for token in vague_tokens:
                if token in intent_l:
                    raise ValueError(
                        f"plan.plotting.plot_intents[{i}] is too vague ('{intent_s}'). Use concrete field-based intents only."
                    )

            if intent_l not in preferred_intents:
                raise ValueError(
                    f"plan.plotting.plot_intents[{i}] is not an allowed direct intent ('{intent_s}')."
                )

            if intent_l in blocked_direct_intents:
                raise ValueError(
                    f"plan.plotting.plot_intents[{i}] should be avoided for this workflow ('{intent_s}')."
                )

            referenced_fields = []
            for field_name in sorted(allowed_direct_plot_fields | allowed_derived_plot_fields, key=len, reverse=True):
                if field_name.lower() in intent_l:
                    referenced_fields.append(field_name)

            if len(referenced_fields) == 0:
                raise ValueError(
                    f"plan.plotting.plot_intents[{i}] must reference at least one concrete retrieved or derived field. Got '{intent_s}'."
                )

            for field_name in referenced_fields:
                if field_name in allowed_derived_plot_fields:
                    continue
                if field_name not in retrieved_fields:
                    raise ValueError(
                        f"plan.plotting.plot_intents[{i}] references field '{field_name}' that is not present in plan.retrieval.fields."
                    )

        if not isinstance(plan.get("documentation"), dict):
            raise ValueError("plan.documentation must be a dict.")
        documentation = plan["documentation"]
        if documentation.get("enabled") is not True:
            raise ValueError("plan.documentation.enabled must be True.")
        if not isinstance(documentation.get("template"), str) or not documentation["template"].strip():
            raise ValueError("plan.documentation.template must be a non-empty string.")
        if documentation.get("output") != "PDF":
            raise ValueError('plan.documentation.output must be "PDF".')

        for s in steps:
            if "inputs" in s and not (
                isinstance(s["inputs"], list) and all(isinstance(x, str) and x.strip() for x in s["inputs"])
            ):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'inputs' (must be list[str]).")
            if "outputs" in s and not (
                isinstance(s["outputs"], list) and all(isinstance(x, str) and x.strip() for x in s["outputs"])
            ):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'outputs' (must be list[str]).")
            if "constraints" in s and not isinstance(s["constraints"], dict):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'constraints' (must be dict).")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in planner_tool: invalid plan format: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plan"] = plan
    context_variables["plan_description"] = plan_description.strip()
    context_variables["success_criteria"] = [x.strip() for x in success_criteria]
    context_variables["plan_score"] = plan_score_f
    context_variables["planner_status"] = "completed"

    next_agent = plan["routing"]["after_planner"]
    context_variables["next_agent"] = next_agent

    message = {
        "plan_description": context_variables["plan_description"],
        "success_criteria": context_variables["success_criteria"],
        "plan_score": context_variables["plan_score"],
        "plan": plan,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False, default=str),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool Reviewer — review_plan

def review_plan_tool(
    decision: Annotated[
        Literal["approve", "revise", "handoff"],
        "approve -> go to retriever; revise -> go back to task improver; handoff -> go back to human.",
    ],
    review_notes: Annotated[str, "Short, concrete notes about what is wrong/right in the plan."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlanReviewer"
    plan = context_variables.get("plan")

    fallback_next = "AgentTaskImprover"
    if isinstance(plan, dict):
        routing_safe = plan.get("routing")
        if isinstance(routing_safe, dict):
            candidate = routing_safe.get("on_revise", fallback_next)
            if isinstance(candidate, str) and candidate.strip():
                fallback_next = candidate.strip()

    if context_variables.get("plan_review_status") == "completed":
        existing_review_notes = context_variables.get("review_notes", "")
        existing_review_status = context_variables.get("review_status", None)

        next_agent_existing = fallback_next
        if isinstance(plan, dict):
            routing_existing = plan.get("routing")
            if isinstance(routing_existing, dict):
                if existing_review_status == "approved":
                    candidate = routing_existing.get("on_approve", next_agent_existing)
                    if isinstance(candidate, str) and candidate.strip():
                        next_agent_existing = candidate.strip()
                elif existing_review_status == "handoff":
                    candidate = routing_existing.get("on_handoff", next_agent_existing)
                    if isinstance(candidate, str) and candidate.strip():
                        next_agent_existing = candidate.strip()
                else:
                    candidate = routing_existing.get("on_revise", next_agent_existing)
                    if isinstance(candidate, str) and candidate.strip():
                        next_agent_existing = candidate.strip()

        context_variables["next_agent"] = next_agent_existing

        if existing_review_status == "approved":
            message_prefix = "Plan review -> approved"
        elif existing_review_status == "handoff":
            message_prefix = "Plan review -> handoff"
        else:
            message_prefix = "Plan review -> revise"

        return ReplyResult(
            message=f"{message_prefix}\n{existing_review_notes}",
            target=AgentNameTarget(next_agent_existing),
            context_variables=context_variables,
        )

    context_variables["plan_review_status"] = "running"

    try:
        if plan is None or not isinstance(plan, dict):
            raise ValueError("Missing or invalid plan in context_variables['plan'].")
        if decision not in {"approve", "revise", "handoff"}:
            raise ValueError("decision must be one of: approve, revise, handoff.")
        if not isinstance(review_notes, str) or not review_notes.strip():
            raise ValueError("review_notes must be a non-empty string.")
    except ValueError as e:
        context_variables["plan_review_status"] = "failed"
        context_variables["review_status"] = "revise"
        context_variables["review_notes"] = f"{str(e)}"
        context_variables["next_agent"] = fallback_next
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(fallback_next),
            context_variables=context_variables,
        )

    try:
        if not isinstance(plan.get("schema_version"), str) or not plan["schema_version"].strip():
            raise ValueError("plan.schema_version must be a non-empty string.")

        if not isinstance(plan.get("main_task_improved"), str) or not plan["main_task_improved"].strip():
            raise ValueError("plan.main_task_improved must be a non-empty string.")

        routing = plan.get("routing")
        if not isinstance(routing, dict):
            raise ValueError("plan.routing must be a dict.")

        required_routing = (
            "after_planner",
            "on_approve",
            "on_revise",
            "on_handoff",
            "after_plot_suggest",
            "after_plot_select",
            "after_code",
            "after_plot_validate",
            "on_plot_validation_failed",
            "after_latex_write",
            "after_latex_compile",
            "on_latex_compile_failed",
        )
        for k in required_routing:
            if not isinstance(routing.get(k), str) or not routing[k].strip():
                raise ValueError(f"plan.routing.{k} must be a non-empty string.")

        if routing["on_approve"].strip() != "AgentMaterialsRetriever":
            raise ValueError('plan.routing.on_approve must be "AgentMaterialsRetriever".')
        if routing["on_revise"].strip() != "AgentTaskImprover":
            raise ValueError('plan.routing.on_revise must be "AgentTaskImprover".')
        if routing["on_handoff"].strip() != "Human":
            raise ValueError('plan.routing.on_handoff must be "Human".')

        steps = plan.get("steps")
        if not isinstance(steps, list) or len(steps) == 0:
            raise ValueError("plan.steps must be a non-empty list.")

        steps_by_id = {}
        for s in steps:
            if not isinstance(s, dict):
                raise ValueError("Each step in plan.steps must be a dict.")
            sid = s.get("id")
            if not isinstance(sid, str) or not sid.strip():
                raise ValueError("Each step must have a non-empty 'id'.")
            if not isinstance(s.get("agent"), str) or not s["agent"].strip():
                raise ValueError(f"Step '{sid}' must have a non-empty 'agent'.")
            steps_by_id[sid.strip()] = s

        required_steps = [
            "retrieve",
            "analyze",
            "plot_suggest",
            "plot_select",
            "plot_generate_v1",
            "plot_validate",
            "plot_generate_v2",
            "latex_write",
            "latex_compile",
        ]
        missing_required_steps = [x for x in required_steps if x not in steps_by_id]
        if missing_required_steps:
            raise ValueError(f"Missing required step(s): {missing_required_steps}")

        strict_agents = {
            "retrieve": "AgentMaterialsRetriever",
            "analyze": "AgentAnalyzer",
            "plot_suggest": "AgentPlotSuggester",
            "plot_select": "AgentPlotSelector",
            "plot_generate_v1": "AgentCoder",
            "plot_validate": "AgentPlotValidator",
            "plot_generate_v2": "AgentCoder",
            "latex_write": "AgentLatexWriter",
            "latex_compile": "AgentLatexCompiler",
        }
        for step_id, expected_agent in strict_agents.items():
            actual_agent = steps_by_id[step_id].get("agent")
            if actual_agent != expected_agent:
                raise ValueError(f"Step '{step_id}' must use agent '{expected_agent}'.")

        retrieval = plan.get("retrieval")
        if not isinstance(retrieval, dict):
            raise ValueError("plan.retrieval must be a dict.")
        if not isinstance(retrieval.get("search_criteria"), dict):
            raise ValueError("plan.retrieval.search_criteria must be a dict.")

        fields = retrieval.get("fields")
        if not isinstance(fields, list) or not fields or not all(isinstance(x, str) and x.strip() for x in fields):
            raise ValueError("plan.retrieval.fields must be a non-empty list of non-empty strings.")

        sn = retrieval.get("sample_number")
        if not isinstance(sn, int) or sn <= 0:
            raise ValueError("plan.retrieval.sample_number must be a positive int.")

        plotting = plan.get("plotting", None)
        if plotting is not None and not isinstance(plotting, dict):
            raise ValueError("plan.plotting must be a dict if present.")
        if isinstance(plotting, dict):
            enabled = plotting.get("enabled", None)
            if not isinstance(enabled, bool):
                raise ValueError("plan.plotting.enabled must be a bool.")
            if enabled is True:
                n_candidates = plotting.get("n_candidates", None)
                n_selected = plotting.get("n_selected", None)
                output_dir = plotting.get("output_dir", None)
                plot_intents = plotting.get("plot_intents", None)

                if not isinstance(n_candidates, int) or n_candidates != 10:
                    raise ValueError("plan.plotting.n_candidates must be int == 10.")
                if not isinstance(n_selected, int) or n_selected != 5:
                    raise ValueError("plan.plotting.n_selected must be int == 5.")
                if not isinstance(output_dir, str) or output_dir.strip() != "plots":
                    raise ValueError('plan.plotting.output_dir must be "plots".')
                if not isinstance(plot_intents, list) or not plot_intents or not all(isinstance(x, str) and x.strip() for x in plot_intents):
                    raise ValueError("plan.plotting.plot_intents must be a non-empty list of non-empty strings.")

        documentation = plan.get("documentation", None)
        if documentation is not None and not isinstance(documentation, dict):
            raise ValueError("plan.documentation must be a dict if present.")
        if isinstance(documentation, dict):
            if not isinstance(documentation.get("enabled"), bool):
                raise ValueError("plan.documentation.enabled must be a bool.")
            if documentation["enabled"]:
                if not isinstance(documentation.get("template"), str) or not documentation["template"].strip():
                    raise ValueError("plan.documentation.template must be a non-empty string.")
                if documentation.get("output") != "PDF":
                    raise ValueError('plan.documentation.output must be "PDF".')

    except ValueError as e:
        context_variables["plan_review_status"] = "failed"
        context_variables["review_status"] = "revise"
        context_variables["review_notes"] = f"{review_notes.strip()} | validation_error: {e}"
        next_agent = plan["routing"]["on_revise"] if isinstance(plan.get("routing"), dict) else "AgentTaskImprover"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    context_variables["review_notes"] = review_notes.strip()

    if decision == "approve":
        next_agent = plan["routing"]["on_approve"]
        context_variables["review_status"] = "approved"
        context_variables["plan_review_status"] = "completed"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> approved\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    if decision == "revise":
        next_agent = plan["routing"]["on_revise"]
        context_variables["review_status"] = "revise"
        context_variables["plan_review_status"] = "completed"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    next_agent = plan["routing"]["on_handoff"]
    context_variables["review_status"] = "handoff"
    context_variables["plan_review_status"] = "completed"
    context_variables["next_agent"] = next_agent
    return ReplyResult(
        message=f"Plan review -> handoff\n{context_variables['review_notes']}",
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — materials_project_retriever

def material_retriever(
    context_variables: ContextVariables,
    search_criteria: Optional[dict] = None,
    fields: Optional[List[str]] = None,
    sample_number: Optional[int] = None,
    plan: Optional[dict] = None,
) -> ReplyResult:
    """Retrieve materials from Materials Project using plan.retrieval.search_criteria / fields / sample_number."""

    agent_name = "AgentMaterialsRetriever"

    if plan is not None:
        if isinstance(plan, dict) and plan:
            context_variables["plan"] = plan
        else:
            context_variables["next_agent"] = "AgentPlanner"
            return ReplyResult(
                message="Error in material_retriever: plan must be a non-empty dict if provided.",
                target=AgentNameTarget("AgentPlanner"),
                context_variables=context_variables,
            )

    plan = context_variables.get("plan", {})
    if not isinstance(plan, dict):
        plan = {}

    routing = plan.get("routing", {})
    if not isinstance(routing, dict):
        routing = {}
    next_on_revise = routing.get("on_revise", "AgentPlanner")

    retrieval = plan.get("retrieval", {})
    if not isinstance(retrieval, dict):
        retrieval = {}

    try:
        if not isinstance(plan, dict) or not plan:
            raise ValueError("Missing or invalid plan in context_variables['plan'].")
        if not isinstance(retrieval, dict) or not retrieval:
            raise ValueError("Missing plan['retrieval'] dict.")

        search_criteria = retrieval.get("search_criteria", None)
        fields = retrieval.get("fields", None)
        sample_number = retrieval.get("sample_number", None)

        if not isinstance(search_criteria, dict):
            raise ValueError("plan['retrieval']['search_criteria'] must be a dict.")
        if not isinstance(fields, list) or len(fields) == 0 or not all(isinstance(f, str) and f.strip() for f in fields):
            raise ValueError("plan['retrieval']['fields'] must be a non-empty list of non-empty strings.")
        if not isinstance(sample_number, int) or sample_number <= 0:
            raise ValueError("plan['retrieval']['sample_number'] must be a positive int.")
    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = {}
        context_variables["fields"] = []
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    guardrails = plan.get("guardrails", {}) if isinstance(plan, dict) else {}
    sn_guard = guardrails.get("sample_number") if isinstance(guardrails, dict) else None

    if isinstance(sn_guard, dict):
        sn_min = sn_guard.get("min")
        sn_max = sn_guard.get("max")
        if isinstance(sn_min, int) and sample_number < sn_min:
            sample_number = sn_min
        if isinstance(sn_max, int) and sample_number > sn_max:
            sample_number = sn_max

    allowed_search_keys = None
    allowed_fields = None

    if isinstance(guardrails, dict):
        ak = guardrails.get("allowed_search_keys")
        af = guardrails.get("allowed_fields")
        if isinstance(ak, list) and all(isinstance(x, str) and x.strip() for x in ak):
            allowed_search_keys = set(x.strip() for x in ak)
        if isinstance(af, list) and all(isinstance(x, str) and x.strip() for x in af):
            allowed_fields = set(x.strip() for x in af)

    if allowed_search_keys is None:
        allowed_search_keys = {
            "band_gap",
            "energy_above_hull",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
            "elements",
            "chemsys",
            "exclude_elements",
        }

    if allowed_fields is None:
        allowed_fields = {
            "material_id",
            "formula_pretty",
            "band_gap",
            "energy_above_hull",
            "density",
            "volume",
            "symmetry",
            "nsites",
            "elements",
            "chemsys",
            "is_stable",
            "bulk_modulus",
            "shear_modulus",
        }

    try:
        if not all(isinstance(k, str) and k.strip() for k in search_criteria.keys()):
            raise ValueError("search_criteria keys must be non-empty strings.")

        if "excluded_elements" in search_criteria:
            raise ValueError(
                "Use 'exclude_elements' instead of 'excluded_elements' in plan['retrieval']['search_criteria']."
            )

        bad_keys = [k for k in search_criteria.keys() if k not in allowed_search_keys]
        if bad_keys:
            raise ValueError(
                f"Unsupported search_criteria key(s): {bad_keys}. "
                f"Allowed keys: {sorted(list(allowed_search_keys))}"
            )

        bad_fields = [f for f in fields if f not in allowed_fields]
        if bad_fields:
            raise ValueError(
                f"Unsupported field(s): {bad_fields}. "
                f"Allowed fields: {sorted(list(allowed_fields))}"
            )

    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    alias_map = {
        "num_sites": "nsites",
        "k_voigt": "bulk_modulus",
        "g_voigt": "shear_modulus",
        "pretty_formula": "formula_pretty",
    }

    def _is_range_tuple(v) -> bool:
        return isinstance(v, tuple) and len(v) == 2

    def _normalize_numeric_range(v):
        if _is_range_tuple(v):
            lo, hi = v
            if lo is None or hi is None:
                raise ValueError("Numeric range filters must be (min, max) with both values set.")
            if not isinstance(lo, (int, float)) or not isinstance(hi, (int, float)):
                raise ValueError("Numeric range filters must be (min, max) numbers.")
            if lo > hi:
                raise ValueError("Numeric range filters must satisfy min <= max.")
            return (lo, hi)

        if isinstance(v, (int, float)):
            return (v, v)

        return v

    normalized_criteria = {}
    try:
        for key, value in search_criteria.items():
            mapped_key = alias_map.get(key, key)

            if mapped_key in {"band_gap", "energy_above_hull", "density", "nsites", "bulk_modulus", "shear_modulus"}:
                normalized_criteria[mapped_key] = _normalize_numeric_range(value)
                continue

            if mapped_key in {"elements", "chemsys", "exclude_elements"}:
                if not (
                    isinstance(value, list)
                    and len(value) > 0
                    and all(isinstance(x, str) and x.strip() for x in value)
                ):
                    raise ValueError(f"{key} must be a non-empty list of non-empty strings.")
                normalized_criteria[mapped_key] = [x.strip() for x in value]
                continue

            normalized_criteria[mapped_key] = value
    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    api_key = os.getenv("MP_API_KEY")
    if not api_key:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message="Error in material_retriever: Missing MP_API_KEY environment variable.",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    try:
        with MPRester(api_key) as mpr:
            all_results = list(
                mpr.materials.summary.search(
                    fields=fields,
                    **normalized_criteria,
                )
            )
    except Exception as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever while querying Materials Project: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    if not all_results:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message="No materials found for the given search_criteria. Revise filters while keeping the main intent.",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )


    results = all_results[:sample_number]

    def _jsonable(x):
        if x is None:
            return None
        if isinstance(x, (str, int, float, bool)):
            return x
        if isinstance(x, (list, tuple)):
            return [_jsonable(v) for v in x]
        if isinstance(x, dict):
            return {str(k): _jsonable(v) for k, v in x.items()}
        if hasattr(x, "model_dump"):
            try:
                return _jsonable(x.model_dump())
            except Exception:
                pass
        if hasattr(x, "dict"):
            try:
                return _jsonable(x.dict())
            except Exception:
                pass
        if hasattr(x, "__dict__"):
            try:
                return _jsonable(vars(x))
            except Exception:
                pass
        return str(x)

    serialized_results = [_jsonable(x) for x in results]

    retrieval_request_path = os.path.join(PATHS["data_processed"], "retrieval_request.json")
    retrieval_results_path = os.path.join(PATHS["data_raw"], "mp_results_raw.json")

    with open(retrieval_request_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "search_criteria": search_criteria,
                "normalized_criteria": normalized_criteria,
                "fields": fields,
                "sample_number_requested": sample_number,
                "sample_number_returned": len(results),
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    with open(retrieval_results_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "mp_results": serialized_results,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    next_agent = "AgentAnalyzer"
    steps = plan.get("steps")
    if isinstance(steps, list):
        for s in steps:
            if isinstance(s, dict) and s.get("id") == "analyze":
                a = s.get("agent")
                if isinstance(a, str) and a.strip():
                    next_agent = a.strip()
                break

    context_variables["search_criteria"] = search_criteria
    context_variables["fields"] = fields
    context_variables["sample_number"] = len(results)
    context_variables["mp_results"] = results
    context_variables["retrieval_request_path"] = retrieval_request_path
    context_variables["retrieval_results_path"] = retrieval_results_path
    context_variables["next_agent"] = next_agent

    return ReplyResult(
        message=f"Retrieved {len(results)} materials from Materials Project with search_criteria={search_criteria}.",
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — final_conclusion_tool (Analyzer)

def final_conclusion_tool(
    context_variables: ContextVariables,
    final_conclusion: str | None = None,
) -> ReplyResult:
    agent_name = "AgentAnalyzer"
    results = context_variables.get("mp_results", None)
    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    def _reply_error(message: str) -> ReplyResult:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=json.dumps({"error": message}, ensure_ascii=False),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if results is None or results == []:
        return _reply_error('No materials available in context_variables["mp_results"].')

    if not isinstance(results, list):
        return _reply_error('context_variables["mp_results"] must be a list.')

    results = [r for r in results if r is not None]
    if len(results) == 0:
        return _reply_error('context_variables["mp_results"] contains no valid entries.')

    if not isinstance(final_conclusion, str) or not final_conclusion.strip():
        return _reply_error("final_conclusion must be a non-empty string generated by AgentAnalyzer.")

    main_task = context_variables.get("main_task_improved", "")
    if not isinstance(main_task, str):
        main_task = ""

    def _to_float(x):
        try:
            if x is None:
                return None
            return float(x)
        except Exception:
            return None

    def _get_value(entry, key):
        if isinstance(entry, dict):
            return entry.get(key)
        return getattr(entry, key, None)

    def _percentiles(vals, ps):
        if not vals:
            return {p: None for p in ps}
        s = sorted(vals)
        n = len(s)
        out = {}
        for p in ps:
            if n == 1:
                out[p] = s[0]
                continue
            q = (p / 100.0) * (n - 1)
            lo = int(q)
            hi = min(lo + 1, n - 1)
            frac = q - lo
            out[p] = s[lo] * (1 - frac) + s[hi] * frac
        return out

    def _jsonable(x):
        if x is None:
            return None
        if isinstance(x, (str, int, float, bool)):
            return x
        if isinstance(x, (list, tuple)):
            return [_jsonable(v) for v in x]
        if isinstance(x, dict):
            return {str(k): _jsonable(v) for k, v in x.items()}
        if hasattr(x, "model_dump"):
            try:
                return _jsonable(x.model_dump())
            except Exception:
                pass
        if hasattr(x, "as_dict"):
            try:
                return _jsonable(x.as_dict())
            except Exception:
                pass
        if hasattr(x, "__dict__"):
            try:
                return _jsonable(vars(x))
            except Exception:
                pass
        try:
            return str(x)
        except Exception:
            return None

    def _as_material_dict(x):
        if isinstance(x, dict):
            d = dict(x)
        elif hasattr(x, "model_dump"):
            try:
                d = x.model_dump()
            except Exception:
                d = {}
        elif hasattr(x, "dict"):
            try:
                d = x.dict()
            except Exception:
                d = {}
        else:
            keys = [
                "material_id",
                "formula_pretty",
                "band_gap",
                "energy_above_hull",
                "e_above_hull",
                "density",
                "elements",
                "is_stable",
            ]
            d = {}
            for k in keys:
                try:
                    d[k] = getattr(x, k, None)
                except Exception:
                    d[k] = None
        return _jsonable(d)

    def _contains_any(text: str, patterns: list[str]) -> str | None:
        lowered = text.lower()
        for pattern in patterns:
            if pattern in lowered:
                return pattern
        return None

    serializable_results = [_as_material_dict(r) for r in results]

    bg_vals = []
    eah_vals = []
    dens_vals = []

    bg_missing = 0
    eah_missing = 0
    dens_missing = 0

    for entry in serializable_results:
        bg = _to_float(_get_value(entry, "band_gap"))
        eah_raw = _get_value(entry, "energy_above_hull")
        if eah_raw is None:
            eah_raw = _get_value(entry, "e_above_hull")
        eah = _to_float(eah_raw)
        dens = _to_float(_get_value(entry, "density"))

        bg_vals.append(bg)
        eah_vals.append(eah)
        dens_vals.append(dens)

        if bg is None:
            bg_missing += 1
        if eah is None:
            eah_missing += 1
        if dens is None:
            dens_missing += 1

    bg_clean = [v for v in bg_vals if isinstance(v, (int, float))]
    eah_clean = [v for v in eah_vals if isinstance(v, (int, float))]
    dens_clean = [v for v in dens_vals if isinstance(v, (int, float))]

    n_total = len(serializable_results)

    stable_count = 0
    near_stable_count = 0
    meta_count = 0
    unstable_count = 0

    for v in eah_clean:
        if v <= 1e-6:
            stable_count += 1
        elif v <= 0.05:
            near_stable_count += 1
        elif v <= 0.2:
            meta_count += 1
        else:
            unstable_count += 1

    bg_bins = {
        "gapless_or_metal_like": 0,
        "narrow_gap": 0,
        "semiconductor": 0,
        "wide_bandgap": 0,
        "very_wide_bandgap": 0,
    }

    for v in bg_clean:
        if v < 0.1:
            bg_bins["gapless_or_metal_like"] += 1
        elif v < 1.0:
            bg_bins["narrow_gap"] += 1
        elif v < 3.0:
            bg_bins["semiconductor"] += 1
        elif v <= 6.0:
            bg_bins["wide_bandgap"] += 1
        else:
            bg_bins["very_wide_bandgap"] += 1

    def _validate_final_conclusion(text: str) -> str | None:
        lowered = text.lower()

        forbidden_global_claims = [
            "all materials meet",
            "all retrieved materials meet",
            "all candidates meet",
            "materials meet the criteria",
            "retrieved materials meet the criteria",
            "candidates meet the criteria",
            "materials satisfy the criteria",
            "retrieved materials satisfy the criteria",
            "candidates satisfy the criteria",
            "materials are suitable",
            "retrieved materials are suitable",
            "candidates are suitable",
            "the retrieved materials are suitable",
            "the materials are suitable",
            "all materials are suitable",
            "all retrieved materials are suitable",
            "matching the requested criteria",
        ]

        matched = _contains_any(lowered, forbidden_global_claims)
        if matched is not None:
            return f"final_conclusion contains an unsupported suitability/compliance claim ('{matched}')."

        if eah_missing > 0:
            matched = _contains_any(
                lowered,
                [
                    "all retrieved materials have energy_above_hull",
                    "all materials have energy_above_hull",
                    f"all {n_total} materials",
                ],
            )
            if matched is not None:
                return f"final_conclusion overstates energy_above_hull completeness ('{matched}')."

        if bg_missing > 0:
            matched = _contains_any(
                lowered,
                [
                    "all retrieved materials have band_gap",
                    "all materials have band_gap",
                ],
            )
            if matched is not None:
                return f"final_conclusion overstates band_gap completeness ('{matched}')."

        if dens_missing > 0:
            matched = _contains_any(
                lowered,
                [
                    "all retrieved materials have density",
                    "all materials have density",
                ],
            )
            if matched is not None:
                return f"final_conclusion overstates density completeness ('{matched}')."

        if "predominantly wide" in lowered or "mostly wide" in lowered:
            bg_available = sum(bg_bins.values())
            if bg_available == 0 or (bg_bins["wide_bandgap"] + bg_bins["very_wide_bandgap"]) <= (bg_available / 2):
                return "final_conclusion claims band gaps are predominantly wide, but the structured summary does not support that."

        if "densities mostly fall within the requested range" in lowered or "densities mostly fall in the requested range" in lowered:
            if len(dens_clean) == 0:
                return "final_conclusion refers to density range, but density is unavailable for all entries."

        if "energy above hull values are concentrated at stable or near-stable levels" in lowered:
            if len(eah_clean) == 0 or (stable_count + near_stable_count) <= 0:
                return "final_conclusion claims stable or near-stable concentration, but available energy_above_hull data does not support that."

        return None

    concise = final_conclusion.strip()
    validation_error = _validate_final_conclusion(concise)
    if validation_error is not None:
        return _reply_error(f"final_conclusion is not consistent with analysis_summary: {validation_error}")

    bg_p = _percentiles(bg_clean, [10, 50, 90])
    eah_p = _percentiles(eah_clean, [10, 50, 90])
    dens_p = _percentiles(dens_clean, [10, 50, 90])

    output = {
        "schema_version": "1.1",
        "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "main_task_improved": main_task.strip(),
        "counts": {
            "materials_retrieved": n_total,
            "missing": {
                "band_gap": bg_missing,
                "energy_above_hull": eah_missing,
                "density": dens_missing,
            },
        },
        "ranges": {
            "band_gap_eV": {
                "p10": bg_p[10],
                "p50": bg_p[50],
                "p90": bg_p[90],
            },
            "energy_above_hull_eV_per_atom": {
                "p10": eah_p[10],
                "p50": eah_p[50],
                "p90": eah_p[90],
            },
            "density_g_cm3": {
                "p10": dens_p[10],
                "p50": dens_p[50],
                "p90": dens_p[90],
            },
        },
        "patterns": {
            "stability_bins": {
                "stable_eah_eq_0": stable_count,
                "near_stable_le_0p05": near_stable_count,
                "metastable_0p05_to_0p2": meta_count,
                "likely_unstable_gt_0p2": unstable_count,
            },
            "band_gap_bins": bg_bins,
        },
        "limitations": [
            "Summary uses only retrieved fields (band_gap, energy_above_hull, density).",
            "No thermal transport, melting point, oxidation kinetics, toxicity, or mechanical validation included.",
        ],
    }

    context_variables["analysis_summary"] = output
    context_variables["final_conclusion"] = concise

    plotting_enabled = False
    if isinstance(plan, dict):
        plotting = plan.get("plotting", {})
        if isinstance(plotting, dict):
            plotting_enabled = bool(plotting.get("enabled", False))

    if plotting_enabled:
        next_agent = "AgentPlotSuggester"
        steps = plan.get("steps", [])
        if isinstance(steps, list):
            for step in steps:
                if isinstance(step, dict) and step.get("id") == "plot_suggest":
                    candidate = step.get("agent")
                    if isinstance(candidate, str) and candidate.strip():
                        next_agent = candidate.strip()
                    break
    else:
        next_agent = routing.get("after_analyzer", None)
        if not isinstance(next_agent, str) or not next_agent.strip():
            next_agent = "AgentLatexWriter"

    context_variables["next_agent"] = next_agent

    final_path = os.path.join(PATHS["data_processed"], "final_conclusion.json")
    materials_path = os.path.join(PATHS["data_processed"], "materials_data.json")

    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "analysis_summary": output,
                "final_conclusion": concise,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    with open(materials_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "mp_results": serializable_results,
                "analysis_summary": output,
                "final_conclusion": concise,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    context_variables["final_conclusion_path"] = final_path
    context_variables["materials_data_path"] = materials_path

    return ReplyResult(
        message=json.dumps(
            {
                "analysis_summary": output,
                "final_conclusion": concise,
            },
            indent=2,
            ensure_ascii=False,
        ),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — plot_suggester_tool

def plot_suggester_tool(
    candidates: Annotated[List[dict], "List of plot candidate dicts (must be length 10)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotSuggester"

    _MAX_SUGGESTER_RETRIES = 2
    _suggester_retries = int(context_variables.get("suggester_retries", 0))

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}
    next_agent = routing.get("after_plot_suggest", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentPlotSelector"

    def _error(message: str) -> ReplyResult:
        nonlocal _suggester_retries
        _suggester_retries += 1
        context_variables["suggester_retries"] = _suggester_retries
        context_variables["next_agent"] = agent_name
        context_variables["plot_candidates_error"] = message
        context_variables["plot_suggest_status"] = "failed"
        if _suggester_retries > _MAX_SUGGESTER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in plot_suggester_tool: AgentPlotSuggester exceeded {_MAX_SUGGESTER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        return ReplyResult(
            message=f"Error in plot_suggester_tool: {message}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        if not isinstance(candidates, list):
            return _error(
                "candidates must be a list of 10 items. Rebuild and resubmit one complete valid set of exactly 10 candidates."
            )

        if len(candidates) != 10:
            return _error(
                f"candidates must be a list of 10 items. Got {len(candidates)}."
            )

        retrieval = plan.get("retrieval", {}) if isinstance(plan, dict) else {}
        retrieval_fields = retrieval.get("fields", []) if isinstance(retrieval, dict) else []
        if not isinstance(retrieval_fields, list):
            retrieval_fields = []

        available_direct_fields = {x.strip() for x in retrieval_fields if isinstance(x, str) and x.strip()}
        allowed_direct_fields = {"band_gap", "density", "energy_above_hull"}
        available_direct_fields = {x for x in available_direct_fields if x in allowed_direct_fields}

        if available_direct_fields != allowed_direct_fields:
            return _error(
                f"Current run must expose exactly these direct numeric fields for plotting: {sorted(allowed_direct_fields)}. "
                f"Got {sorted(available_direct_fields)} from plan.retrieval.fields."
            )

        forbidden_fields = {
            "stability_bin",
            "band_gap_bin",
            "stability_bucket",
            "band_gap_bucket",
            "counts",
            "material_count",
            "number_of_materials",
            "materials_count",
            "stability_bins",
            "band_gap_bins",
            "count",
            "frequency",
            "elements",
            "formula",
            "formula_pretty",
            "material_id",
            "spacegroup",
            "symmetry",
            "structure",
            "composition",
            "is_stable",
            "volume",
            "bulk_modulus",
            "shear_modulus",
            "chemsys",
            "nsites",
            "k_voigt",
            "g_voigt",
        }

        required_keys = ["plot_id", "title", "description", "data_requirements", "axes", "rationale", "plot_type"]
        expected_ids = [f"plot_{i}" for i in range(1, 11)]

        seen_ids = set()
        normalized = []

        for i, item in enumerate(candidates):
            if not isinstance(item, dict):
                return _error(f"candidates[{i}] must be a dict.")

            for k in required_keys:
                if k not in item:
                    return _error(f"candidates[{i}] missing required key '{k}'.")

            plot_id = item.get("plot_id")
            title = item.get("title")
            description = item.get("description")
            data_requirements = item.get("data_requirements")
            axes = item.get("axes")
            rationale = item.get("rationale")
            plot_type = item.get("plot_type")

            if not isinstance(plot_id, str) or not plot_id.strip():
                return _error(f"candidates[{i}].plot_id must be a non-empty string.")
            plot_id_s = plot_id.strip()

            if plot_id_s != expected_ids[i]:
                return _error(
                    f"candidates[{i}].plot_id must be exactly '{expected_ids[i]}', got '{plot_id_s}'."
                )

            if plot_id_s in seen_ids:
                return _error(f"Duplicate plot_id found: {plot_id_s}.")
            seen_ids.add(plot_id_s)

            if not isinstance(title, str) or not title.strip():
                return _error(f"candidates[{i}].title must be a non-empty string.")
            if not isinstance(description, str) or not description.strip():
                return _error(f"candidates[{i}].description must be a non-empty string.")
            if not isinstance(rationale, str) or not rationale.strip():
                return _error(f"candidates[{i}].rationale must be a non-empty string.")

            if not isinstance(data_requirements, list) or not data_requirements:
                return _error(f"candidates[{i}].data_requirements must be a non-empty list.")

            cleaned_requirements = []
            for x in data_requirements:
                if not isinstance(x, str) or not x.strip():
                    return _error(f"candidates[{i}].data_requirements must contain non-empty strings only.")
                field_name = x.strip()
                if field_name in forbidden_fields:
                    return _error(
                        f"candidates[{i}].data_requirements contains forbidden field '{field_name}'."
                    )
                if field_name not in allowed_direct_fields:
                    return _error(
                        f"candidates[{i}].data_requirements contains unavailable field '{field_name}'. "
                        f"Allowed fields: {sorted(allowed_direct_fields)}."
                    )
                cleaned_requirements.append(field_name)

            cleaned_requirements = list(dict.fromkeys(cleaned_requirements))

            if not isinstance(axes, dict):
                return _error(f"candidates[{i}].axes must be a dict.")

            required_axis_keys = {"type", "x", "y", "z", "intensity"}
            if set(axes.keys()) != required_axis_keys:
                return _error(
                    f"candidates[{i}].axes must contain exactly keys {sorted(required_axis_keys)}."
                )

            try:
                candidate_model = PlotCandidate(
                    plot_id=plot_id_s,
                    title=title.strip(),
                    plot_type=plot_type,
                    data_requirements=cleaned_requirements,
                    axes=axes,
                )
                candidate_model.validate_structure()
            except Exception as e:
                return _error(f"candidates[{i}] invalid structure: {e}")

            normalized_axes = candidate_model.axes.model_dump()
            normalized_plot_type = candidate_model.plot_type.value

            if normalized_plot_type not in {"scatter_2d", "histogram", "scatter_3d"}:
                return _error(
                    f"candidates[{i}] uses unsupported plot_type '{normalized_plot_type}'."
                )

            axis_values = {}
            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = normalized_axes.get(axis_key)
                if axis_value is None:
                    axis_values[axis_key] = None
                    continue
                if not isinstance(axis_value, str) or not axis_value.strip():
                    return _error(f"candidates[{i}].axes.{axis_key} must be null or a non-empty string.")
                axis_value_s = axis_value.strip()
                if axis_value_s in forbidden_fields:
                    return _error(
                        f"candidates[{i}].axes.{axis_key} uses forbidden field '{axis_value_s}'."
                    )
                if axis_value_s not in allowed_direct_fields:
                    return _error(
                        f"candidates[{i}].axes.{axis_key} uses unavailable field '{axis_value_s}'. "
                        f"Allowed fields: {sorted(allowed_direct_fields)}."
                    )
                axis_values[axis_key] = axis_value_s

            if normalized_plot_type == "scatter_2d":
                if normalized_axes.get("type") != "2d":
                    return _error(f"candidates[{i}] scatter_2d must use axes.type='2d'.")
                if axis_values["x"] is None or axis_values["y"] is None:
                    return _error(f"candidates[{i}] scatter_2d requires non-null x and y axes.")
                if axis_values["z"] is not None or axis_values["intensity"] is not None:
                    return _error(f"candidates[{i}] scatter_2d must use null z and intensity axes.")

            if normalized_plot_type == "histogram":
                if normalized_axes.get("type") != "hist":
                    return _error(f"candidates[{i}] histogram must use axes.type='hist'.")
                if axis_values["x"] is None:
                    return _error(f"candidates[{i}] histogram requires non-null x axis.")
                if axis_values["y"] is not None or axis_values["z"] is not None or axis_values["intensity"] is not None:
                    return _error(f"candidates[{i}] histogram must use only x axis.")

            if normalized_plot_type == "scatter_3d":
                if normalized_axes.get("type") != "3d":
                    return _error(f"candidates[{i}] scatter_3d must use axes.type='3d'.")
                if axis_values["x"] is None or axis_values["y"] is None or axis_values["z"] is None:
                    return _error(f"candidates[{i}] scatter_3d requires non-null x, y, and z axes.")
                if axis_values["intensity"] is not None:
                    return _error(f"candidates[{i}] scatter_3d must use null intensity axis.")

            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = axis_values[axis_key]
                if axis_value is not None and axis_value not in cleaned_requirements:
                    return _error(
                        f"candidates[{i}].axes.{axis_key}='{axis_value}' must appear in data_requirements."
                    )

            normalized.append(
                {
                    "plot_id": plot_id_s,
                    "title": title.strip(),
                    "description": description.strip(),
                    "data_requirements": cleaned_requirements,
                    "axes": normalized_axes,
                    "rationale": rationale.strip(),
                    "plot_type": normalized_plot_type,
                }
            )

        plot_types = [x["plot_type"] for x in normalized]
        if any(x not in {"scatter_2d", "histogram", "scatter_3d"} for x in plot_types):
            return _error("All candidates must be scatter_2d, histogram, or scatter_3d only.")

        hist_count = sum(1 for x in normalized if x["plot_type"] == "histogram")
        if hist_count < 3:
            return _error("At least 3 histogram candidates are required in this run.")

    except ValueError as e:
        return _error(str(e))

    plot_candidates_path = os.path.join(PATHS["data_processed"], "plot_candidates.json")

    with open(plot_candidates_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "plot_candidates_count": len(normalized),
                "plot_candidates": normalized,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    context_variables["plot_candidates"] = normalized
    context_variables["plot_candidates_path"] = plot_candidates_path
    context_variables["plot_candidates_error"] = ""
    context_variables["plot_suggest_status"] = "completed"
    context_variables["suggester_retries"] = 0
    context_variables["next_agent"] = next_agent

    message = {
        "plot_candidates_count": len(normalized),
        "plot_candidates": normalized,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )


# Tool — plot_selector_tool

def plot_selector_tool(
    selected_plot_ids: Annotated[List[str], "List of selected plot_id values (must select exactly 5)."],
    selection_notes: Annotated[str, "Short notes explaining why these plots were selected."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotSelector"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent = routing.get("after_plot_select", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentCoder"

    suggester_agent = "AgentPlotSuggester"
    steps = plan.get("steps", []) if isinstance(plan, dict) else []
    if isinstance(steps, list):
        for step in steps:
            if isinstance(step, dict) and step.get("id") == "plot_suggest":
                candidate_agent = step.get("agent")
                if isinstance(candidate_agent, str) and candidate_agent.strip():
                    suggester_agent = candidate_agent.strip()
                    break

    if context_variables.get("plot_select_status") == "completed" and context_variables.get("plot_selected"):
        context_variables["next_agent"] = next_agent
        message = {
            "plot_selected_ids": [x.get("plot_id") for x in context_variables.get("plot_selected", []) if isinstance(x, dict)],
            "plot_selected": context_variables.get("plot_selected", []),
            "selection_notes": context_variables.get("plot_selection_notes", ""),
        }
        return ReplyResult(
            message=json.dumps(message, indent=2, ensure_ascii=False),
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    context_variables["plot_select_status"] = "running"

    def _fail(message: str, target: str | None = None) -> ReplyResult:
        fail_target = target or agent_name
        context_variables["plot_select_status"] = "failed"
        context_variables["next_agent"] = fail_target
        return ReplyResult(
            message=f"Error in plot_selector_tool: {message}",
            target=AgentNameTarget(fail_target),
            context_variables=context_variables,
        )

    try:
        if not isinstance(selected_plot_ids, list) or len(selected_plot_ids) == 0:
            raise ValueError("selected_plot_ids must be a non-empty list.")
        if not all(isinstance(x, str) and x.strip() for x in selected_plot_ids):
            raise ValueError("selected_plot_ids must contain non-empty strings only.")
        if not isinstance(selection_notes, str) or not selection_notes.strip():
            raise ValueError("selection_notes must be a non-empty string.")
    except ValueError as e:
        return _fail(str(e))

    candidates = context_variables.get("plot_candidates", None)
    if not isinstance(candidates, list) or len(candidates) == 0:
        return _fail(
            "Missing or invalid context_variables['plot_candidates'] (must be a non-empty list).",
            target=suggester_agent,
        )

    try:
        if len(candidates) != 10:
            raise ValueError(f"Expected exactly 10 plot_candidates, got {len(candidates)}.")

        retrieval = plan.get("retrieval", {}) if isinstance(plan, dict) else {}
        retrieval_fields = retrieval.get("fields", []) if isinstance(retrieval, dict) else []
        if not isinstance(retrieval_fields, list):
            retrieval_fields = []

        available_direct_fields = {x.strip() for x in retrieval_fields if isinstance(x, str) and x.strip()}
        allowed_direct_fields = {"band_gap", "density", "energy_above_hull"}
        available_direct_fields = {x for x in available_direct_fields if x in allowed_direct_fields}

        id_to_candidate = {}
        for c in candidates:
            if not isinstance(c, dict):
                raise ValueError("Each plot candidate must be a dict.")

            pid = c.get("plot_id")
            title = c.get("title")
            description = c.get("description")
            data_requirements = c.get("data_requirements")
            axes = c.get("axes")
            plot_type = c.get("plot_type")

            if not isinstance(pid, str) or not pid.strip():
                raise ValueError("Each plot candidate must have a non-empty 'plot_id'.")
            if not isinstance(title, str) or not title.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a non-empty 'title'.")
            if not isinstance(description, str) or not description.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a non-empty 'description'.")
            if not isinstance(data_requirements, list) or not data_requirements or not all(
                isinstance(x, str) and x.strip() for x in data_requirements
            ):
                raise ValueError(f"Plot candidate '{pid}' must have valid data_requirements.")
            if not isinstance(axes, dict):
                raise ValueError(f"Plot candidate '{pid}' must have valid axes.")
            if not isinstance(plot_type, str) or not plot_type.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a valid plot_type.")
            if pid in id_to_candidate:
                raise ValueError(f"Duplicate plot_id found: {pid}")

            id_to_candidate[pid.strip()] = {
                "plot_id": pid.strip(),
                "title": title.strip(),
                "description": description.strip(),
                "data_requirements": [x.strip() for x in data_requirements],
                "axes": {
                    "type": axes.get("type"),
                    "x": axes.get("x"),
                    "y": axes.get("y"),
                    "z": axes.get("z"),
                    "intensity": axes.get("intensity"),
                },
                "plot_type": plot_type.strip(),
                "rationale": c.get("rationale", "").strip() if isinstance(c.get("rationale"), str) else "",
            }

        plotting = plan.get("plotting", {}) if isinstance(plan, dict) else {}
        n_selected = plotting.get("n_selected", 5)
        if not isinstance(n_selected, int) or n_selected <= 0:
            n_selected = 5

        ids_normalized = [x.strip() for x in selected_plot_ids]

        if len(set(ids_normalized)) != len(ids_normalized):
            raise ValueError("Duplicate plot_id values are not allowed.")

        if len(ids_normalized) != n_selected:
            raise ValueError(f"Must select exactly {n_selected} plot_id values.")

        missing = [pid for pid in ids_normalized if pid not in id_to_candidate]
        if missing:
            raise ValueError(f"Selected plot_id not found in candidates: {missing}")

        selected = [id_to_candidate[pid] for pid in ids_normalized]

        robust_plot_types = {"scatter_2d", "histogram", "scatter_3d"}

        for item in selected:
            pid = item["plot_id"]
            data_requirements = item["data_requirements"]
            axes = item["axes"]
            plot_type = item["plot_type"]

            if plot_type not in robust_plot_types:
                raise ValueError(f"Selected plot '{pid}' has unsupported plot_type '{plot_type}'.")

            for field_name in data_requirements:
                if field_name not in available_direct_fields:
                    raise ValueError(
                        f"Selected plot '{pid}' uses unavailable data_requirement '{field_name}'. "
                        f"Available direct fields: {sorted(available_direct_fields)}."
                    )

            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = axes.get(axis_key)
                if axis_value is None:
                    continue
                if axis_value not in available_direct_fields:
                    raise ValueError(
                        f"Selected plot '{pid}' uses unavailable axes.{axis_key}='{axis_value}'. "
                        f"Available direct fields: {sorted(available_direct_fields)}."
                    )
                if axis_value not in data_requirements:
                    raise ValueError(
                        f"Selected plot '{pid}' has axes.{axis_key}='{axis_value}' not present in data_requirements."
                    )

            if plot_type == "scatter_2d":
                if axes.get("type") != "2d":
                    raise ValueError(f"Selected plot '{pid}' must have axes.type='2d' for scatter_2d.")
                if axes.get("x") is None or axes.get("y") is None:
                    raise ValueError(f"Selected plot '{pid}' must define x and y axes for scatter_2d.")
                if axes.get("z") is not None or axes.get("intensity") is not None:
                    raise ValueError(f"Selected plot '{pid}' must not define z or intensity for scatter_2d.")
                if len(data_requirements) != 2:
                    raise ValueError(f"Selected plot '{pid}' must use exactly 2 data_requirements for scatter_2d.")

            if plot_type == "histogram":
                if axes.get("type") != "hist":
                    raise ValueError(f"Selected plot '{pid}' must have axes.type='hist' for histogram.")
                if axes.get("x") is None:
                    raise ValueError(f"Selected plot '{pid}' must define x axis for histogram.")
                if axes.get("y") is not None or axes.get("z") is not None or axes.get("intensity") is not None:
                    raise ValueError(f"Selected plot '{pid}' must only define x axis for histogram.")
                if len(data_requirements) != 1:
                    raise ValueError(f"Selected plot '{pid}' must use exactly 1 data_requirement for histogram.")

            if plot_type == "scatter_3d":
                if axes.get("type") != "3d":
                    raise ValueError(f"Selected plot '{pid}' must have axes.type='3d' for scatter_3d.")
                if axes.get("x") is None or axes.get("y") is None or axes.get("z") is None:
                    raise ValueError(f"Selected plot '{pid}' must define x, y, and z axes for scatter_3d.")
                if axes.get("intensity") is not None:
                    raise ValueError(f"Selected plot '{pid}' must not define intensity for scatter_3d.")
                if len(data_requirements) != 3:
                    raise ValueError(f"Selected plot '{pid}' must use exactly 3 data_requirements for scatter_3d.")

    except ValueError as e:
        return _fail(str(e))

    plot_selected_path = os.path.join(PATHS["data_processed"], "plot_selected.json")

    with open(plot_selected_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "plot_selected_ids": ids_normalized,
                "plot_selected": selected,
                "selection_notes": selection_notes.strip(),
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    context_variables["plot_selected"] = selected
    context_variables["plot_selected_ids"] = ids_normalized
    context_variables["plot_selected_path"] = plot_selected_path
    context_variables["plot_selection_notes"] = selection_notes.strip()
    context_variables["plot_select_status"] = "completed"
    context_variables["next_agent"] = next_agent

    message = {
        "plot_selected_ids": ids_normalized,
        "plot_selected": selected,
        "selection_notes": context_variables["plot_selection_notes"],
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )


# Tool — python_coder

project_folder = os.environ.get("PROJECT_FOLDER", os.path.abspath("ag2_project"))
os.makedirs(project_folder, exist_ok=True)
os.environ["PROJECT_FOLDER"] = project_folder


class PythonCode(BaseModel):
    code: str = Field(..., description="Full python code executed by the coder agent")


def _safe_filename(name: str) -> str:
    name = os.path.basename((name or "").strip())
    if len(name) == 0:
        return "script.py"

    keep = []
    for ch in name:
        if ch.isalnum() or ch in {".", "_", "-"}:
            keep.append(ch)
    name = "".join(keep)

    if len(name) == 0:
        name = "script.py"
    if not name.endswith(".py"):
        name = name + ".py"
    if len(name) > 120:
        name = name[-120:]
    return name


def _extract_plot_id_from_png_name(filename: str) -> str:
    if not isinstance(filename, str):
        return ""
    match = re.match(r"^(plot_\d+)_", os.path.basename(filename))
    if match:
        return match.group(1)
    return ""


def _code_has_plot_branch(code_str: str, plot_type: str) -> bool:
    single = f"plot_type == '{plot_type}'"
    double = f'plot_type == "{plot_type}"'
    return single in code_str or double in code_str


def _has_empty_elif_block(code_str: str) -> bool:
    lines = code_str.splitlines()
    for i, line in enumerate(lines[:-1]):
        stripped = line.strip()
        if stripped.startswith("elif ") and stripped.endswith(":"):
            next_line = lines[i + 1]
            if len(next_line.strip()) == 0:
                return True
            if len(next_line) - len(next_line.lstrip()) <= len(line) - len(line.lstrip()):
                return True
    return False


def _count_plot_type_branches(code_str: str, plot_type: str) -> int:
    single = f"plot_type == '{plot_type}'"
    double = f'plot_type == "{plot_type}"'
    return code_str.count(single) + code_str.count(double)


def _extract_branch_block(code_str: str, plot_type: str) -> str:
    lines = code_str.splitlines()
    branch_headers = [
        f"if plot_type == '{plot_type}':",
        f'if plot_type == "{plot_type}":',
        f"elif plot_type == '{plot_type}':",
        f'elif plot_type == "{plot_type}":',
    ]

    start_idx = None
    start_indent = None

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped in branch_headers:
            start_idx = i
            start_indent = len(line) - len(line.lstrip())
            break

    if start_idx is None:
        return ""

    block_lines = [lines[start_idx]]
    for j in range(start_idx + 1, len(lines)):
        line = lines[j]
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())

        if stripped and indent <= start_indent and (
            stripped.startswith("elif ")
            or stripped.startswith("else:")
            or stripped.startswith("if ")
        ):
            break

        block_lines.append(line)

    return "\n".join(block_lines)


def _has_fragile_histogram_pattern(code_str: str) -> bool:
    if "plot_type == 'histogram'" not in code_str and 'plot_type == "histogram"' not in code_str:
        return False

    hist_block = _extract_branch_block(code_str, "histogram")
    if not hist_block:
        return True

    if "plt.hist(" not in hist_block:
        return True

    bad_patterns = [
        "x_data, y_data, z_data, intensity_data = [], [], [], []",
        "y_data = []",
        "z_data = []",
        "intensity_data = []",
        "plt.scatter(",
        "ax.scatter(",
    ]

    if any(pattern in hist_block for pattern in bad_patterns):
        return True

    if "hist_data" not in hist_block:
        return True

    if "axes['x']" not in hist_block and 'axes["x"]' not in hist_block and ".get('x')" not in hist_block and '.get("x")' not in hist_block:
        return True

    return False


def _has_inconsistent_manifest_naming_pattern(code_str: str) -> bool:
    bad_patterns = [
        'filename = f"{title',
        "filename = f'{title",
        'name": title.lower(',
        "name': title.lower(",
        'path": os.path.join("plots", title.lower(',
        "path': os.path.join('plots', title.lower(",
        'path": "plots/" + title.lower(',
        "path': 'plots/' + title.lower(",
        'filename = title.lower(',
        "filename = title.lower(",
        '"name": title',
        "'name': title",
    ]
    return any(pattern in code_str for pattern in bad_patterns)


def _has_manifest_comprehension_pattern(code_str: str) -> bool:
    bad_patterns = [
        "plots_v1 = [{",
        "plots_v2 = [{",
        '"plots_v1": [{',
        '"plots_v2": [{',
        "'plots_v1': [{",
        "'plots_v2': [{",
        "for plot_spec in plot_selected]",
        "for spec in plot_selected]",
        "for plot in plot_selected]",
    ]
    return any(pattern in code_str for pattern in bad_patterns)


def _has_stable_placeholder_pattern(code_str: str) -> bool:
    if "Insufficient data" not in code_str:
        return False

    save_markers = [
        "plt.savefig(out_path)",
        "savefig(out_path)",
    ]
    close_markers = [
        "plt.close()",
        "close()",
    ]
    exists_markers = [
        "assert os.path.exists(out_path)",
        "os.path.exists(out_path)",
    ]

    has_save = any(marker in code_str for marker in save_markers)
    has_close = any(marker in code_str for marker in close_markers)
    has_exists = any(marker in code_str for marker in exists_markers)

    return has_save and has_close and has_exists


def _is_overly_complex_plot_script(code_str: str) -> bool:
    metrics = 0

    if code_str.count("try:") >= 3:
        metrics += 1
    if code_str.count("except ") >= 3:
        metrics += 1
    if code_str.count("def ") >= 5:
        metrics += 1
    if code_str.count("for plot_spec in plot_selected") >= 2:
        metrics += 1
    if code_str.count("for spec in plot_selected") >= 2:
        metrics += 1
    if code_str.count("for plot in plot_selected") >= 2:
        metrics += 1
    if code_str.count("plt.figure(") >= 8:
        metrics += 1
    if len(code_str.splitlines()) >= 260:
        metrics += 1

    return metrics >= 2


def _hardcodes_regeneration_state(code_str: str) -> bool:
    bad_patterns = [
        "plots_to_regenerate =",
        "plots_to_regenerate=",
        "regenerate_mode = True",
        "regenerate_mode=True",
        "regenerate_mode = False",
        "regenerate_mode=False",
        "is_v2 = True",
        "is_v2=True",
        "is_v2 = False",
        "is_v2=False",
    ]
    return any(pattern in code_str for pattern in bad_patterns)


def _uses_wrong_formula_field(code_str: str) -> bool:
    return "pretty_formula" in code_str


def _has_json_null_token(code_str: str) -> bool:
    return re.search(r"\bnull\b", code_str) is not None


def python_coder_tool(
    code: Annotated[str, "A single block of Python code to execute."],
    file_name: Annotated[str, "Name of the code file to store the executed script, e.g., 'script.py'"],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentCoder"

    _MAX_CODER_RETRIES = 5
    _coder_retries = int(context_variables.get("coder_retries", 0))

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent_v1 = routing.get("after_code", None)
    if not isinstance(next_agent_v1, str) or not next_agent_v1.strip():
        next_agent_v1 = "Human"

    next_agent_v2 = routing.get("after_plot_validate", None)
    if not isinstance(next_agent_v2, str) or not next_agent_v2.strip():
        next_agent_v2 = next_agent_v1

    selected_specs = context_variables.get("plot_selected", [])
    plots_to_regenerate_raw = context_variables.get("plots_to_regenerate", [])

    plots_to_regenerate = []
    if isinstance(plots_to_regenerate_raw, list):
        for x in plots_to_regenerate_raw:
            if isinstance(x, str) and x.strip():
                plots_to_regenerate.append(x.strip())

    if len(set(plots_to_regenerate)) != len(plots_to_regenerate):
        plots_to_regenerate = list(dict.fromkeys(plots_to_regenerate))

    is_v2 = len(plots_to_regenerate) > 0

    def _fail_to_coder(message: str, increment_retry: bool = False) -> ReplyResult:
        nonlocal _coder_retries

        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"

        if increment_retry:
            _coder_retries += 1
            context_variables["coder_retries"] = _coder_retries
            if _coder_retries > _MAX_CODER_RETRIES:
                return ReplyResult(
                    message=(
                        f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                        "Handing off to Human for manual intervention."
                    ),
                    target=AgentNameTarget("Human"),
                    context_variables=context_variables,
                )

        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=message,
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        if not isinstance(code, str) or len(code.strip()) == 0:
            raise ValueError("The 'code' parameter must be a non-empty string.")
        if not isinstance(file_name, str) or len(file_name.strip()) == 0:
            raise ValueError("The 'file_name' parameter must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in python_coder_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    safe_name = _safe_filename(file_name)

    try:
        os.environ["PLOT_SELECTED_JSON"] = json.dumps(
            selected_specs,
            ensure_ascii=False,
            default=str,
        )
    except Exception:
        os.environ["PLOT_SELECTED_JSON"] = "[]"

    try:
        os.environ["PLOTS_TO_REGENERATE_JSON"] = json.dumps(
            plots_to_regenerate,
            ensure_ascii=False,
            default=str,
        )
    except Exception:
        os.environ["PLOTS_TO_REGENERATE_JSON"] = "[]"

    code_stripped = code.strip()

    forbidden_plot_selected_patterns = [
        "plot_selected =",
        "plot_selected=",
        "plot_selected = [",
        "plot_selected=[",
        "plot_selected = list(",
        "plot_selected=list(",
        "plot_selected = json.loads(",
        "plot_selected=json.loads(",
    ]

    if "plot_selected" not in code_stripped:
        return _fail_to_coder(
            "Error in python_coder_tool: generated code does not reference 'plot_selected'. "
            "Plots must be generated by iterating over the runtime plot_selected variable.",
            increment_retry=True,
        )

    if any(pattern in code_stripped for pattern in forbidden_plot_selected_patterns):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code redefines 'plot_selected'. "
            "The runtime variable 'plot_selected' is the single source of truth and must be used directly.",
            increment_retry=True,
        )

    if "plot_selected[0]" in code_stripped:
        return _fail_to_coder(
            "Error in python_coder_tool: generated code uses 'plot_selected[0]' "
            "inside the loop. Always use the current iteration variable instead of indexing plot_selected directly.",
            increment_retry=True,
        )

    if _hardcodes_regeneration_state(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code hardcodes regeneration state. "
            "Use the runtime plots_to_regenerate variable directly and infer V1/V2 from it.",
            increment_retry=True,
        )

    if _uses_wrong_formula_field(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code uses 'pretty_formula'. "
            "Use 'formula_pretty' to match the pipeline contract.",
            increment_retry=True,
        )

    if _has_json_null_token(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code contains 'null'. "
            "Use valid Python values only.",
            increment_retry=True,
        )

    selected_plot_types = []
    if isinstance(selected_specs, list):
        for spec in selected_specs:
            if isinstance(spec, dict):
                plot_type = spec.get("plot_type")
                if isinstance(plot_type, str) and plot_type.strip():
                    selected_plot_types.append(plot_type.strip())

    for plot_type in sorted(set(selected_plot_types)):
        if not _code_has_plot_branch(code_stripped, plot_type):
            return _fail_to_coder(
                f"Error in python_coder_tool: generated code does not contain a clear branch for "
                f"plot_type '{plot_type}'.",
                increment_retry=True,
            )

    if _has_empty_elif_block(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code contains an incomplete if/elif plotting branch. "
            "Every if / elif / else branch must have a full indented body.",
            increment_retry=True,
        )

    for plot_type in sorted(set(selected_plot_types)):
        if _count_plot_type_branches(code_stripped, plot_type) > 1:
            return _fail_to_coder(
                f"Error in python_coder_tool: generated code contains multiple branches for plot_type '{plot_type}'. "
                "Use one single stable if/elif chain with exactly one branch per plot type.",
                increment_retry=True,
            )

    if "scatter_3d" in selected_plot_types:
        fragile_3d_patterns = [
            "plt.gca(projection='3d')",
            'plt.gca(projection="3d")',
            "plt.figure().add_subplot(",
        ]
        if any(pattern in code_stripped for pattern in fragile_3d_patterns):
            return _fail_to_coder(
                "Error in python_coder_tool: generated code uses a fragile 3D plotting pattern. "
                "Use only: fig = plt.figure() and ax = fig.add_subplot(111, projection='3d') inside the scatter_3d branch.",
                increment_retry=True,
            )

        if "fig = plt.figure()" not in code_stripped or (
            "ax = fig.add_subplot(111, projection='3d')" not in code_stripped
            and 'ax = fig.add_subplot(111, projection="3d")' not in code_stripped
        ):
            return _fail_to_coder(
                "Error in python_coder_tool: scatter_3d must use an explicit dedicated 3D figure pattern.",
                increment_retry=True,
            )

    if "histogram" in selected_plot_types and _has_fragile_histogram_pattern(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: histogram handling is too fragile. "
            "Histogram must have its own independent branch, must use plt.hist(...), "
            "and must not depend on a shared scatter-style data preparation path.",
            increment_retry=True,
        )

    if _has_inconsistent_manifest_naming_pattern(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: manifest naming is inconsistent with the plot filename contract. "
            "Use one single filename rule only: filename = f\"{plot_id}_{slug}.png\" "
            "and make the manifest name/path match the exact saved file.",
            increment_retry=True,
        )

    if _has_manifest_comprehension_pattern(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: manifest construction is too complex or fragile. "
            "Do not use list comprehensions or one-line manifest builders. "
            "Build manifest entries with an explicit loop and explicit local variables.",
            increment_retry=True,
        )

    if not _has_stable_placeholder_pattern(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated code does not show a stable placeholder strategy. "
            "Use one simple placeholder path with the exact text 'Insufficient data', "
            "the normal out_path saveflow, and the same naming contract as regular plots.",
            increment_retry=True,
        )

    if _is_overly_complex_plot_script(code_stripped):
        return _fail_to_coder(
            "Error in python_coder_tool: generated plotting script is unnecessarily complex. "
            "Reduce complexity and use one stable explicit implementation per plot type, "
            "one explicit placeholder path, and one explicit manifest loop.",
            increment_retry=True,
        )

    context_variables["plots_status"] = "v2_running" if is_v2 else "v1_running"

    expected_plot_ids = []
    if isinstance(selected_specs, list):
        for s in selected_specs:
            if isinstance(s, dict):
                pid = s.get("plot_id")
                if isinstance(pid, str) and pid.strip():
                    expected_plot_ids.append(pid.strip())

    expects_v1_plots = (not is_v2) and isinstance(selected_specs, list) and len(selected_specs) == 5

    if expects_v1_plots:
        if len(expected_plot_ids) != 5:
            return _fail_to_coder(
                "Error in python_coder_tool: plot_selected must contain exactly 5 items with non-empty 'plot_id' values before generating plots.",
                increment_retry=True,
            )
        if len(set(expected_plot_ids)) != len(expected_plot_ids):
            return _fail_to_coder(
                "Error in python_coder_tool: plot_selected contains duplicate plot_id values.",
                increment_retry=True,
            )

    if expects_v1_plots:
        try:
            plots_dir_v1_pre = PATHS["plots_v1"]
            os.makedirs(plots_dir_v1_pre, exist_ok=True)
            for fn in os.listdir(plots_dir_v1_pre):
                if fn.lower().endswith(".png"):
                    try:
                        os.remove(os.path.join(plots_dir_v1_pre, fn))
                    except Exception:
                        pass
        except Exception:
            pass

    if is_v2:
        try:
            plots_dir_v2_pre = PATHS["plots_v2"]
            os.makedirs(plots_dir_v2_pre, exist_ok=True)
            for fn in os.listdir(plots_dir_v2_pre):
                if not fn.lower().endswith(".png"):
                    continue
                pid = _extract_plot_id_from_png_name(fn)
                if pid in plots_to_regenerate:
                    try:
                        os.remove(os.path.join(plots_dir_v2_pre, fn))
                    except Exception:
                        pass
        except Exception:
            pass

    runtime_prelude = (
        "import os\n"
        "import json\n"
        "PROJECT_FOLDER = os.environ.get('PROJECT_FOLDER', 'ag2_project')\n"
        "os.makedirs(PROJECT_FOLDER, exist_ok=True)\n"
        "os.chdir(PROJECT_FOLDER)\n"
        "\n"
        "PATHS = {\n"
        "    'data_raw': os.path.join(PROJECT_FOLDER, 'data', 'raw'),\n"
        "    'data_processed': os.path.join(PROJECT_FOLDER, 'data', 'processed'),\n"
        "    'plots_v1': os.path.join(PROJECT_FOLDER, 'plots', 'v1'),\n"
        "    'plots_v2': os.path.join(PROJECT_FOLDER, 'plots', 'v2'),\n"
        "    'plots_manifests': os.path.join(PROJECT_FOLDER, 'plots', 'manifests'),\n"
        "    'reports_latex': os.path.join(PROJECT_FOLDER, 'reports', 'latex'),\n"
        "    'reports_pdf': os.path.join(PROJECT_FOLDER, 'reports', 'pdf'),\n"
        "    'scripts': os.path.join(PROJECT_FOLDER, 'scripts', 'generated'),\n"
        "    'tmp': os.path.join(PROJECT_FOLDER, 'tmp'),\n"
        "}\n"
        "for _path in PATHS.values():\n"
        "    os.makedirs(_path, exist_ok=True)\n"
        "\n"
        "PLOTS_DIR_V1 = PATHS['plots_v1']\n"
        "PLOTS_DIR_V2 = PATHS['plots_v2']\n"
        "\n"
        "PLOT_SELECTED_JSON = os.environ.get('PLOT_SELECTED_JSON', '[]')\n"
        "try:\n"
        "    plot_selected = json.loads(PLOT_SELECTED_JSON)\n"
        "except Exception:\n"
        "    plot_selected = []\n"
        "\n"
        "PLOTS_TO_REGENERATE_JSON = os.environ.get('PLOTS_TO_REGENERATE_JSON', '[]')\n"
        "try:\n"
        "    plots_to_regenerate = json.loads(PLOTS_TO_REGENERATE_JSON)\n"
        "except Exception:\n"
        "    plots_to_regenerate = []\n"
        "\n"
        "def _ensure_materials_data_json():\n"
        "    materials_path = os.path.join(PATHS['data_processed'], 'materials_data.json')\n"
        "    final_conclusion_path = os.path.join(PATHS['data_processed'], 'final_conclusion.json')\n"
        "    if os.path.exists(materials_path):\n"
        "        return\n"
        "    if not os.path.exists(final_conclusion_path):\n"
        "        return\n"
        "    try:\n"
        "        with open(final_conclusion_path, 'r', encoding='utf-8') as f:\n"
        "            payload = json.load(f)\n"
        "    except Exception:\n"
        "        return\n"
        "    mp_results = payload.get('mp_results')\n"
        "    analysis_summary = payload.get('analysis_summary')\n"
        "    final_conclusion = payload.get('final_conclusion')\n"
        "    if mp_results is None:\n"
        "        return\n"
        "    with open(materials_path, 'w', encoding='utf-8') as f:\n"
        "        json.dump(\n"
        "            {\n"
        "                'mp_results': mp_results,\n"
        "                'analysis_summary': analysis_summary,\n"
        "                'final_conclusion': final_conclusion,\n"
        "            },\n"
        "            f,\n"
        "            indent=2,\n"
        "            ensure_ascii=False,\n"
        "            default=str,\n"
        "        )\n"
        "\n"
        "_ensure_materials_data_json()\n"
    )

    code_sanitized = code.replace("\t", "    ").strip("\n")
    sanitized_lines = []
    for line in code_sanitized.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("def ", "class ", "import ", "from ")):
            sanitized_lines.append(stripped)
        else:
            sanitized_lines.append(line.rstrip())
    code_sanitized = "\n".join(sanitized_lines).strip()

    full_code = "#!/usr/bin/env python3\n" + runtime_prelude + "\n" + code_sanitized + "\n"

    try:
        compile(full_code, safe_name, "exec")
    except Exception as e:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = f"{e}"
        context_variables["last_execution_exit_code"] = 1
        context_variables["last_executed_file"] = ""
        context_variables["next_agent"] = agent_name

        syntax_msg = str(e)
        if "expected an indented block after 'elif' statement" in syntax_msg:
            message = (
                "Error in python_coder_tool: generated code is not valid Python.\n"
                f"{e}\n"
                "The plotting block contains an incomplete elif branch. "
                "Use one stable if/elif/else chain and make every plot_type branch fully indented and self-contained."
            )
        else:
            message = f"Error in python_coder_tool: generated code is not valid Python.\n{e}"

        return ReplyResult(
            message=message,
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    python_code_model = PythonCode(code=full_code)

    script_path = os.path.join(PATHS["scripts"], safe_name)
    try:
        with open(script_path, "w", encoding="utf-8") as f:
            f.write(python_code_model.code)
    except Exception as e:
        return _fail_to_coder(
            f"Error in python_coder_tool: failed to store executed script.\n{e}",
            increment_retry=True,
        )

    executor = LocalCommandLineCodeExecutor(timeout=600, work_dir=project_folder)
    code_block = CodeBlock(language="python", code=full_code)

    try:
        result = executor.execute_code_blocks([code_block])
    except Exception as e:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = f"{e}"
        context_variables["last_execution_exit_code"] = 1
        context_variables["last_executed_file"] = script_path
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message="Code execution failed due to an internal executor error:\n" + f"{e}\n",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    exit_code = result.exit_code
    output = result.output

    execution_log_path = os.path.join(
        PATHS["tmp"],
        f"{os.path.splitext(safe_name)[0]}_execution.json",
    )

    def _atomic_write_json(path: str, obj) -> None:
        tmp_path = path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
        os.replace(tmp_path, path)

    _atomic_write_json(
        execution_log_path,
        {
            "file_name": safe_name,
            "script_path": script_path,
            "exit_code": exit_code,
            "output": output,
            "is_v2": is_v2,
            "plots_to_regenerate": plots_to_regenerate,
        },
    )

    try:
        plot_desc_by_id = {}
        plot_title_by_id = {}
        if isinstance(selected_specs, list):
            for s in selected_specs:
                if isinstance(s, dict):
                    pid = s.get("plot_id")
                    if isinstance(pid, str) and pid.strip():
                        pid_s = pid.strip()
                        title = s.get("title")
                        desc = s.get("description")
                        if isinstance(title, str) and title.strip():
                            plot_title_by_id[pid_s] = title.strip()
                        if isinstance(desc, str) and desc.strip():
                            plot_desc_by_id[pid_s] = desc.strip()

        if exit_code == 0 and expects_v1_plots:
            plots_dir_v1 = PATHS["plots_v1"]
            os.makedirs(plots_dir_v1, exist_ok=True)

            png_files = []
            for fn in os.listdir(plots_dir_v1):
                if fn.lower().endswith(".png"):
                    png_files.append(fn)
            png_files.sort()

            produced_map = {}
            for fn in png_files:
                pid = _extract_plot_id_from_png_name(fn)
                if not pid:
                    context_variables["plots_status"] = "v1_failed"
                    raise ValueError(
                        f"PNG filename does not follow the required '<plot_id>_<slug>.png' contract: {fn}"
                    )
                produced_map.setdefault(pid, []).append(fn)

            if len(png_files) != 5:
                context_variables["plots_status"] = "v1_failed"
                raise ValueError(
                    f"Expected exactly 5 PNGs in '{plots_dir_v1}', but found {len(png_files)}: {png_files}"
                )

            dup = [pid for pid, files in produced_map.items() if len(files) > 1]
            if dup:
                context_variables["plots_status"] = "v1_failed"
                raise ValueError(
                    f"Duplicate plot_id prefixes detected in PNG filenames: {dup}"
                )

            expected_set = set(expected_plot_ids)
            produced_set = set(produced_map.keys())

            if produced_set != expected_set:
                context_variables["plots_status"] = "v1_failed"
                missing = sorted(list(expected_set - produced_set))
                extra = sorted(list(produced_set - expected_set))
                raise ValueError(
                    f"Produced plots do not match plot_selected. missing={missing}, extra={extra}"
                )

            plots_v1 = []
            for plot_id in expected_plot_ids:
                fn = produced_map[plot_id][0]
                rel_path = os.path.relpath(os.path.join(plots_dir_v1, fn), project_folder).replace("\\", "/")
                description = plot_desc_by_id.get(plot_id, "") or plot_title_by_id.get(plot_id, "") or "Plotting V1 artifact."
                plots_v1.append(
                    {
                        "plot_id": plot_id,
                        "name": fn,
                        "path": rel_path,
                        "description": description,
                    }
                )

            context_variables["plots_v1"] = plots_v1
            context_variables["plots_status"] = "v1_ready"

            manifest_path = os.path.join(PATHS["plots_manifests"], "plots_v1_manifest.json")
            _atomic_write_json(manifest_path, {"plots_v1": plots_v1})
            context_variables["plots_v1_manifest_path"] = manifest_path

        if exit_code == 0 and is_v2:
            plots_dir_v2 = PATHS["plots_v2"]
            os.makedirs(plots_dir_v2, exist_ok=True)

            png_files_v2 = []
            for fn in os.listdir(plots_dir_v2):
                if fn.lower().endswith(".png"):
                    png_files_v2.append(fn)
            png_files_v2.sort()

            produced_map = {}
            for fn in png_files_v2:
                pid = _extract_plot_id_from_png_name(fn)
                if not pid:
                    context_variables["plots_status"] = "v2_failed"
                    raise ValueError(
                        f"PNG filename does not follow the required '<plot_id>_<slug>.png' contract: {fn}"
                    )
                if pid in plots_to_regenerate:
                    produced_map.setdefault(pid, []).append(fn)

            missing = [pid for pid in plots_to_regenerate if pid not in produced_map]
            if missing:
                context_variables["plots_status"] = "v2_failed"
                raise ValueError(f"Missing regenerated plots in plots_v2 for plot_ids: {missing}")

            dup = [pid for pid, files in produced_map.items() if len(files) > 1]
            if dup:
                context_variables["plots_status"] = "v2_failed"
                raise ValueError(f"Multiple regenerated PNGs found for plot_ids in plots_v2: {dup}")

            plots_v2 = []
            for pid in plots_to_regenerate:
                fn = produced_map[pid][0]
                rel_path = os.path.relpath(os.path.join(plots_dir_v2, fn), project_folder).replace("\\", "/")
                description = plot_desc_by_id.get(pid, "") or plot_title_by_id.get(pid, "") or "Plotting V2 artifact."
                plots_v2.append(
                    {
                        "plot_id": pid,
                        "name": fn,
                        "path": rel_path,
                        "description": description,
                    }
                )

            context_variables["plots_v2"] = plots_v2
            context_variables["plots_status"] = "v2_ready"

            manifest_path_v2 = os.path.join(PATHS["plots_manifests"], "plots_v2_manifest.json")
            _atomic_write_json(manifest_path_v2, {"plots_v2": plots_v2})
            context_variables["plots_v2_manifest_path"] = manifest_path_v2

    except Exception as e:
        if is_v2 and context_variables.get("plots_status") not in {"v2_failed", "v2_ready"}:
            context_variables["plots_status"] = "v2_failed"
        if (not is_v2) and context_variables.get("plots_status") not in {"v1_failed", "v1_ready"}:
            context_variables["plots_status"] = "v1_failed"
        context_variables["next_agent"] = agent_name
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = output
        context_variables["last_execution_exit_code"] = exit_code
        context_variables["last_executed_file"] = script_path
        return ReplyResult(
            message=f"Plot artifact processing failed:\n{e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["last_executed_code"] = full_code
    context_variables["last_execution_output"] = output
    context_variables["last_execution_exit_code"] = exit_code
    context_variables["last_executed_file"] = script_path
    context_variables["last_execution_log_path"] = execution_log_path

    if exit_code != 0:
        if is_v2 and context_variables.get("plots_status") != "v2_ready":
            context_variables["plots_status"] = "v2_failed"
        if (not is_v2) and context_variables.get("plots_status") != "v1_ready":
            context_variables["plots_status"] = "v1_failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Code execution failed.\n{output}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["coder_retries"] = 0

    if is_v2:
        context_variables["next_agent"] = next_agent_v2
        return ReplyResult(
            message=f"Code executed successfully.\n{output}",
            target=AgentNameTarget(next_agent_v2),
            context_variables=context_variables,
        )

    context_variables["next_agent"] = next_agent_v1
    return ReplyResult(
        message=f"Code executed successfully.\n{output}",
        target=AgentNameTarget(next_agent_v1),
        context_variables=context_variables,
    )

# Tool — plot_validator

def plot_validator_tool(
    validation_notes: Annotated[str, "Short global validation notes about V1 plots."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotValidator"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    try:
        if not isinstance(validation_notes, str) or not validation_notes.strip():
            raise ValueError("validation_notes must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    plots_status = context_variables.get("plots_status", None)
    if plots_status != "v1_ready":
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in plot_validator_tool: plots_status must be 'v1_ready' before validation. "
                f"Got '{plots_status}'."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        plots_v1 = context_variables.get("plots_v1", None)
        if not isinstance(plots_v1, list) or len(plots_v1) == 0:
            raise ValueError("context_variables['plots_v1'] must be a non-empty list.")

        plotting = plan.get("plotting", {}) if isinstance(plan, dict) else {}
        vision_model = plotting.get("vision_model", "o3")
        if not isinstance(vision_model, str) or not vision_model.strip():
            vision_model = "o3"

        reviews = {}
        plots_to_regenerate = []
        regeneration_reasons = {}

        structural_tokens = [
            "missing axis",
            "wrong plot type",
            "empty plot",
            "no data",
            "insufficient data",
            "file missing",
            "corrupted",
            "unreadable",
            "legend missing",
            "axis label missing",
            "title missing",
        ]

        seen_plot_ids = set()

        for item in plots_v1:
            if not isinstance(item, dict):
                raise ValueError("Each plot in plots_v1 must be a dict.")

            meta = validate_plot_artifact_meta(item)

            plot_id = meta.get("plot_id")
            plot_name = meta.get("name")
            plot_description = meta.get("description")
            plot_path = meta.get("path")

            if not isinstance(plot_id, str) or not plot_id.strip():
                raise ValueError("Each plot in plots_v1 must have a non-empty plot_id.")
            if plot_id in seen_plot_ids:
                raise ValueError(f"Duplicate plot_id found in plots_v1: {plot_id}")
            seen_plot_ids.add(plot_id)

            if not isinstance(plot_name, str) or not plot_name.strip():
                raise ValueError(f"Plot '{plot_id}' must have a non-empty name.")
            if not plot_name.startswith(f"{plot_id}_") or not plot_name.lower().endswith(".png"):
                raise ValueError(
                    f"Plot '{plot_id}' has invalid filename '{plot_name}'. "
                    "Expected '<plot_id>_<slug>.png'."
                )

            expected_rel = os.path.relpath(
                os.path.join(PATHS["plots_v1"], plot_name),
                PROJECT_FOLDER,
            ).replace("\\", "/")
            if plot_path.replace("\\", "/") != expected_rel:
                raise ValueError(
                    f"Plot '{plot_id}' has inconsistent path. "
                    f"Expected '{expected_rel}', got '{plot_path}'."
                )

            if not isinstance(plot_description, str) or not plot_description.strip():
                raise ValueError(f"Plot '{plot_id}' must have a non-empty description.")

            res = evaluate_plot(
                plot_name=plot_name,
                plot_description=plot_description,
                model=vision_model,
                plot_path=plot_path,
            )

            reviews[plot_id] = res.model_dump()

            issues = res.issues if isinstance(res.issues, list) else []

            structural_issue = any(
                isinstance(issue, str) and any(
                    token in issue.lower()
                    for token in structural_tokens
                )
                for issue in issues
            )

            should_regenerate = False
            reason_parts = []

            if res.pass_fail is False:
                should_regenerate = True
                reason_parts.append("pass_fail=False")

            if res.severity == "high":
                should_regenerate = True
                reason_parts.append("severity=high")

            if res.severity == "medium" and structural_issue:
                should_regenerate = True
                reason_parts.append("severity=medium_with_structural_issue")

            if should_regenerate:
                plots_to_regenerate.append(plot_id)
                regeneration_reasons[plot_id] = {
                    "severity": res.severity,
                    "pass_fail": res.pass_fail,
                    "structural_issue": structural_issue,
                    "issues": issues,
                    "reason": ", ".join(reason_parts) if reason_parts else "regeneration_required",
                }

        if len(set(plots_to_regenerate)) != len(plots_to_regenerate):
            raise ValueError("plots_to_regenerate contains duplicates.")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )
    except Exception as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plot_reviews"] = reviews
    context_variables["plots_to_regenerate"] = plots_to_regenerate
    context_variables["plot_regeneration_reasons"] = regeneration_reasons
    context_variables["plot_validation_notes"] = validation_notes.strip()

    plot_validation_path = os.path.join(PATHS["plots_manifests"], "plot_validation.json")
    with open(plot_validation_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "validated_plots": list(reviews.keys()),
                "plot_reviews": reviews,
                "plots_to_regenerate": plots_to_regenerate,
                "regeneration_reasons": regeneration_reasons,
                "validation_notes": validation_notes.strip(),
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    context_variables["plot_validation_path"] = plot_validation_path

    if plots_to_regenerate:
        context_variables["plots_status"] = "v1_failed"
        next_agent = routing.get("on_plot_validation_failed", "AgentCoder")
    else:
        context_variables["plots_status"] = "v1_validated"
        next_agent = routing.get("after_plot_validate", "Human")

    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "Human"

    context_variables["next_agent"] = next_agent

    message = {
        "validated_plots": list(reviews.keys()),
        "plots_to_regenerate": plots_to_regenerate,
        "regeneration_reasons": regeneration_reasons,
        "validation_notes": validation_notes.strip(),
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )    

# Tool — latex_writer_tool

def latex_writer_tool(
    latex_sections: Annotated[str, "LaTeX body content (no preamble)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentLatexWriter"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent = routing.get("after_latex_write", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentLatexCompiler"

    def _reply_error(message: str) -> ReplyResult:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in latex_writer_tool: {message}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    def _normalize_name(name: str) -> str:
        base = os.path.basename(str(name).strip())
        return base.lower().replace("-", "_").replace(" ", "_")

    def _build_plot_registry() -> dict:
        registry = {}

        plots_v1 = context_variables.get("plots_v1", [])
        plots_v2 = context_variables.get("plots_v2", [])

        if not isinstance(plots_v1, list):
            plots_v1 = []
        if not isinstance(plots_v2, list):
            plots_v2 = []

        for item in plots_v1:
            if not isinstance(item, dict):
                continue
            plot_id = item.get("plot_id")
            name = item.get("name")
            path = item.get("path")
            if not plot_id or not name or not path:
                continue
            registry[plot_id] = {
                "plot_id": plot_id,
                "name": os.path.basename(name),
                "path": path,
                "source": "v1",
            }

        for item in plots_v2:
            if not isinstance(item, dict):
                continue
            plot_id = item.get("plot_id")
            name = item.get("name")
            path = item.get("path")
            if not plot_id or not name or not path:
                continue
            registry[plot_id] = {
                "plot_id": plot_id,
                "name": os.path.basename(name),
                "path": path,
                "source": "v2",
            }

        return registry

    def _build_name_lookup(registry: dict) -> dict:
        by_exact = {}
        by_normalized = {}

        for item in registry.values():
            real_name = os.path.basename(item["name"])
            by_exact[real_name] = item
            by_normalized[_normalize_name(real_name)] = item

            stem = os.path.splitext(real_name)[0]
            if stem.startswith(f"{item['plot_id']}_"):
                no_prefix = stem[len(item["plot_id"]) + 1:] + ".png"
                by_normalized[_normalize_name(no_prefix)] = item

        return by_exact, by_normalized

    def _extract_includegraphics_targets(text: str) -> list[str]:
        pattern = r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}"
        return re.findall(pattern, text)

    def _replace_includegraphics_names(text: str, mapping: dict) -> str:
        pattern = r"(\\includegraphics(?:\[[^\]]*\])?\{)([^}]+)(\})"

        def repl(match):
            prefix, old_name, suffix = match.groups()
            new_name = mapping.get(old_name, old_name)
            return f"{prefix}{new_name}{suffix}"

        return re.sub(pattern, repl, text)

    try:
        if not isinstance(latex_sections, str) or not latex_sections.strip():
            raise ValueError("latex_sections must be a non-empty string.")

        template = globals().get("LATEX_TEMPLATE")
        if not isinstance(template, str):
            raise ValueError("LATEX_TEMPLATE is missing or invalid (must be a string).")

        if "{section_content}" not in template:
            raise ValueError("LATEX_TEMPLATE must contain '{section_content}' placeholder.")

        bad_markers = [
            "\\documentclass",
            "\\usepackage",
            "\\begin{document}",
            "\\end{document}",
            "\\maketitle",
            "\\title{",
            "\\author{",
            "\\date{",
            "```",
        ]
        for marker in bad_markers:
            if marker in latex_sections:
                raise ValueError(f"latex_sections must not include template or markdown markers (found '{marker}').")

        required_blocks = [
            "\\begin{abstract}",
            "\\end{abstract}",
            "\\section{Introduction}",
            "\\section{Methods}",
            "\\section{Results}",
            "\\section{Discussion}",
            "\\section{Conclusion}",
        ]
        for block in required_blocks:
            if block not in latex_sections:
                raise ValueError(f"latex_sections is missing required content block '{block}'.")

        registry = _build_plot_registry()
        by_exact, by_normalized = _build_name_lookup(registry)

        include_targets = _extract_includegraphics_targets(latex_sections)
        replacement_map = {}
        used_figures = []

        for requested_name in include_targets:
            requested_base = os.path.basename(requested_name.strip())

            if "/" in requested_name or "\\" in requested_name:
                raise ValueError(
                    f"includegraphics must use filename only, without folders: '{requested_name}'"
                )

            matched = by_exact.get(requested_base)
            if matched is None:
                matched = by_normalized.get(_normalize_name(requested_base))

            if matched is None:
                raise ValueError(
                    f"Figure '{requested_base}' not found in plots_v1/plots_v2 manifests."
                )

            real_name = matched["name"]
            real_path = matched["path"]

            if not os.path.exists(real_path):
                raise ValueError(
                    f"Figure file does not exist on disk: '{real_path}'"
                )

            replacement_map[requested_name] = real_name
            used_figures.append({
                "requested_name": requested_base,
                "resolved_name": real_name,
                "resolved_path": real_path,
                "plot_id": matched["plot_id"],
                "source": matched["source"],
            })

        latex_sections = _replace_includegraphics_names(latex_sections, replacement_map)

    except ValueError as e:
        return _reply_error(str(e))

    reports_latex_dir = PATHS["reports_latex"]
    sections_path = os.path.join(reports_latex_dir, "report_sections.tex")
    report_tex_path = os.path.join(reports_latex_dir, "report.tex")
    paper_tex_path = os.path.join(reports_latex_dir, "paper.tex")
    template_snapshot_path = os.path.join(reports_latex_dir, "latex_template_used.tex")

    sections_s = latex_sections.strip() + "\n"

    try:
        os.makedirs(reports_latex_dir, exist_ok=True)

        copied_figure_paths = []
        for fig in used_figures:
            src = fig["resolved_path"]
            dst = os.path.join(reports_latex_dir, fig["resolved_name"])
            shutil.copy2(src, dst)
            copied_figure_paths.append(dst)

        report_tex = template.replace("{section_content}", sections_s)

        with open(sections_path, "w", encoding="utf-8") as f:
            f.write(sections_s)

        with open(report_tex_path, "w", encoding="utf-8") as f:
            f.write(report_tex)

        with open(paper_tex_path, "w", encoding="utf-8") as f:
            f.write(report_tex)

        with open(template_snapshot_path, "w", encoding="utf-8") as f:
            f.write(template)

    except Exception as e:
        return _reply_error(f"while writing files: {e}")

    context_variables["latex_sections"] = sections_s
    context_variables["report_sections_path"] = sections_path
    context_variables["report_tex_path"] = report_tex_path
    context_variables["paper_tex_path"] = paper_tex_path
    context_variables["latex_tex_path"] = paper_tex_path
    context_variables["latex_template_path"] = template_snapshot_path
    context_variables["latex_used_figures"] = used_figures
    context_variables["latex_copied_figure_paths"] = copied_figure_paths
    context_variables["next_agent"] = next_agent

    message = {
        "report_sections_path": sections_path,
        "report_tex_path": report_tex_path,
        "paper_tex_path": paper_tex_path,
        "latex_tex_path": paper_tex_path,
        "latex_template_path": template_snapshot_path,
        "latex_sections_chars": len(sections_s),
        "latex_used_figures": used_figures,
        "latex_copied_figure_paths": copied_figure_paths,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — latex_compiler_tool

def latex_compiler_tool(
    context_variables: ContextVariables,
    latex_engine: Annotated[
        Literal["tectonic", "pdflatex", "xelatex", "lualatex"],
        "LaTeX engine to use for compilation. Default: tectonic.",
    ] = "tectonic",
    passes: Annotated[int, "Number of compilation passes (1-3). Default: 2."] = 2,
) -> ReplyResult:
    agent_name = "AgentLatexCompiler"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent = routing.get("after_latex_compile", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "Human"

    next_agent_fail = routing.get("on_latex_compile_failed", "AgentLatexWriter")
    if not isinstance(next_agent_fail, str) or not next_agent_fail.strip():
        next_agent_fail = "AgentLatexWriter"

    try:
        if latex_engine not in {"tectonic", "pdflatex", "xelatex", "lualatex"}:
            raise ValueError("latex_engine must be one of: tectonic, pdflatex, xelatex, lualatex.")
        if not isinstance(passes, int) or passes < 1 or passes > 3:
            raise ValueError("passes must be an int in [1, 3].")

        project_folder = os.environ.get("PROJECT_FOLDER", "ag2_project")
        if not isinstance(project_folder, str) or not project_folder.strip():
            raise ValueError("PROJECT_FOLDER env var missing/invalid.")

        latex_tex_path = context_variables.get("latex_tex_path", None)
        if not isinstance(latex_tex_path, str) or not latex_tex_path.strip():
            raise ValueError("Missing or invalid context_variables['latex_tex_path'].")

        if not os.path.isabs(latex_tex_path):
            tex_path_abs = os.path.abspath(os.path.join(project_folder, latex_tex_path))
        else:
            tex_path_abs = os.path.abspath(latex_tex_path)

        reports_latex_abs = os.path.abspath(PATHS["reports_latex"])
        reports_pdf_abs = os.path.abspath(PATHS["reports_pdf"])

        if not tex_path_abs.startswith(reports_latex_abs):
            raise ValueError("latex_tex_path must be inside PATHS['reports_latex'].")

        if not os.path.exists(tex_path_abs):
            raise ValueError(f"LaTeX source not found at '{tex_path_abs}'.")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in latex_compiler_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    requested_engine = latex_engine
    effective_engine = latex_engine

    if shutil.which(effective_engine) is None:
        if effective_engine != "tectonic" and shutil.which("tectonic") is not None:
            effective_engine = "tectonic"
        else:
            context_variables["latex_status"] = "failed"
            context_variables["latex_compile_success"] = False
            context_variables["latex_engine_requested"] = requested_engine
            context_variables["latex_engine"] = effective_engine
            context_variables["next_agent"] = next_agent_fail
            return ReplyResult(
                message=(
                    "LaTeX compilation failed.\n"
                    f"Requested engine '{requested_engine}' is not available in PATH."
                ),
                target=AgentNameTarget(next_agent_fail),
                context_variables=context_variables,
            )

    context_variables["latex_tex_path_abs"] = tex_path_abs

    os.makedirs(PATHS["reports_latex"], exist_ok=True)
    os.makedirs(PATHS["reports_pdf"], exist_ok=True)

    # --- FIX: ensure figures exist in compile directory ---
    try:
        used_figures = context_variables.get("latex_used_figures", [])
        for fig in used_figures:
            src = fig.get("resolved_path")
            name = fig.get("resolved_name")
            if not src or not name:
                continue
            dst = os.path.join(PATHS["reports_latex"], name)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
    except Exception:
        pass
    # --- END FIX ---

    compile_timeout = 300
    compile_workdir = PATHS["reports_latex"]
    executor = LocalCommandLineCodeExecutor(timeout=compile_timeout, work_dir=compile_workdir)

    tex_filename = os.path.basename(tex_path_abs)
    pdf_filename = os.path.splitext(tex_filename)[0] + ".pdf"
    log_filename = os.path.splitext(tex_filename)[0] + ".log"

    if effective_engine == "tectonic":
        cmd = f'tectonic --keep-logs --keep-intermediates "{tex_filename}"'
    else:
        cmd = (
            f"{effective_engine} "
            f"-interaction=nonstopmode "
            f"-halt-on-error "
            f"-file-line-error "
            f"\"{tex_filename}\""
        )

    output_all = []
    exit_code_final = 0

    try:
        for _ in range(passes):
            cb = CodeBlock(language="bash", code=cmd)
            res = executor.execute_code_blocks([cb])
            exit_code_final = res.exit_code
            output_all.append(res.output or "")
            if exit_code_final != 0:
                break
            if effective_engine == "tectonic":
                break

    except Exception as e:
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine_requested"] = requested_engine
        context_variables["latex_engine"] = effective_engine
        context_variables["latex_compile_error"] = f"{e}"
        context_variables["next_agent"] = next_agent_fail
        return ReplyResult(
            message=f"Error in latex_compiler_tool: internal executor error: {e}",
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    combined_output = "\n".join([x for x in output_all if x]).strip()

    pdf_path_latex_dir = os.path.join(PATHS["reports_latex"], pdf_filename)
    pdf_path_final = os.path.join(PATHS["reports_pdf"], pdf_filename)
    log_path = os.path.join(PATHS["reports_latex"], log_filename)

    if exit_code_final != 0:
        log_tail = ""
        try:
            if os.path.exists(log_path):
                with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
                    txt = f.read()
                log_tail = txt[-4000:]
        except Exception:
            log_tail = ""

        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine_requested"] = requested_engine
        context_variables["latex_engine"] = effective_engine
        context_variables["latex_passes"] = passes
        context_variables["latex_compile_output"] = combined_output
        context_variables["latex_log_path"] = log_path if os.path.exists(log_path) else ""
        context_variables["latex_log_tail"] = log_tail
        context_variables["next_agent"] = next_agent_fail

        return ReplyResult(
            message=(
                "LaTeX compilation failed.\n"
                f"requested_engine={requested_engine}, engine={effective_engine}, passes={passes}, exit_code={exit_code_final}\n"
                + (f"\n--- output ---\n{combined_output}\n" if combined_output else "")
                + (f"\n--- log tail ---\n{log_tail}\n" if log_tail else "")
            ),
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    if not os.path.exists(pdf_path_latex_dir):
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine_requested"] = requested_engine
        context_variables["latex_engine"] = effective_engine
        context_variables["latex_passes"] = passes
        context_variables["latex_compile_output"] = combined_output
        context_variables["next_agent"] = next_agent_fail
        return ReplyResult(
            message=(
                "Error in latex_compiler_tool: compilation finished but PDF was not found.\n"
                f"requested_engine={requested_engine}, engine={effective_engine}, expected_pdf='{pdf_path_latex_dir}'."
            ),
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    try:
        shutil.copy2(pdf_path_latex_dir, pdf_path_final)
    except Exception as e:
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine_requested"] = requested_engine
        context_variables["latex_engine"] = effective_engine
        context_variables["latex_passes"] = passes
        context_variables["latex_compile_output"] = combined_output
        context_variables["next_agent"] = next_agent_fail
        return ReplyResult(
            message=f"Error in latex_compiler_tool: compiled PDF exists but could not be copied to reports/pdf: {e}",
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    context_variables["latex_status"] = "ready"
    context_variables["latex_compile_success"] = True
    context_variables["latex_engine_requested"] = requested_engine
    context_variables["latex_engine"] = effective_engine
    context_variables["latex_passes"] = passes
    context_variables["latex_compile_output"] = combined_output
    context_variables["latex_pdf_path"] = pdf_path_final
    context_variables["report_pdf_path"] = pdf_path_final
    context_variables["latex_log_path"] = log_path if os.path.exists(log_path) else ""
    context_variables["next_agent"] = next_agent

    message = {
        "latex_status": "ready",
        "latex_pdf_path": pdf_path_final,
        "latex_tex_path": tex_path_abs,
        "latex_engine_requested": requested_engine,
        "latex_engine": effective_engine,
        "passes": passes,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — handoff_to_human

def handoff_to_human_tool(
    message: Annotated[str, "Short explanation of why the pipeline needs human intervention."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "Human"

    try:
        if not isinstance(message, str) or not message.strip():
            raise ValueError("handoff message must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in handoff_to_human_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    msg = message.strip()
    context_variables["handoff_reason"] = msg
    context_variables["next_agent"] = agent_name

    return ReplyResult(
        message=msg,
        target=AgentNameTarget(agent_name),
        context_variables=context_variables,
    )


_required_tools = [
    "task_improver_tool",
    "planner_tool",
    "review_plan_tool",
    "material_retriever",
    "final_conclusion_tool",
    "plot_suggester_tool",
    "plot_selector_tool",
    "python_coder_tool",
    "plot_validator_tool",
    "latex_writer_tool",
    "latex_compiler_tool",
    "handoff_to_human_tool",
]

_missing = [
    t for t in _required_tools
    if t not in globals() or not callable(globals()[t])
]

if _missing:
    raise RuntimeError(f"Tools missing or not callable: {_missing}")

print(f"✓ Tools loaded OK ({len(_required_tools)}).")

✓ Tools loaded OK (12).


# 7. AGENTS

In [12]:


# Agent: Task Improver

task_improver_message = """
You are AgentTaskImprover.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything outside the tool call.
- Do not add status text.
- Your only valid output is exactly one call to task_improver_tool.

ROLE
- Improve the wording of the user's main task without changing meaning.

HIGHEST-PRIORITY RULE
- Preserve the user's exact objective, constraints, exclusions, thresholds, environmental context, requested workflow elements, exact field names, and critical wording.
- Zero drift is allowed in exclusions, numeric ranges, field names, and critical wording.

MANDATORY PRESERVATION RULES
- Do not add, remove, broaden, narrow, reinterpret, or substitute any user requirement.
- Do not add inferred constraints, implied safety rules, hidden assumptions, indirect compounds, standards, or domain expansions.
- Do not convert exact requirements into softer summaries.
- Do not change chemical or materials constraints.
- Do not change numeric ranges or threshold wording.
- Do not change excluded elements or exclusion phrasing in substance.
- Preserve exact field names when present in the user query, especially:
  - energy_above_hull
  - band_gap
  - density
- Preserve explicit phrases exactly when present, especially:
  - stable
  - excluding toxic elements such as Pb, Cd, and Hg
  - Include plotting in the workflow.
- Do not introduce new deliverables, rankings, output formats, plot types, file formats, report formats, or evaluation criteria.

ALLOWED IMPROVEMENTS
- Minimal wording cleanup only.
- Light grammar cleanup only if meaning remains exactly the same.
- Remove redundancy only if nothing about scope or constraints changes.
- If the task is already clear, keep improved_task near-verbatim.
- If there is any doubt, copy the original wording exactly.

WHEN INFORMATION IS MISSING
- Mention missing information only in analysis or improvements.
- Do not silently fill missing information into improved_task.

OUTPUT CONTRACT
- You must provide:
  - analysis
  - improvements
  - improved_task
- improved_task must stay semantically equivalent to the original task.
- improved_task must not introduce any new requirement.
- improved_task must not remove any original requirement.
- improved_task must preserve exact field names and explicit workflow wording when present.
- For tasks with explicit numeric ranges, exclusions, and plotting instructions, improved_task should usually be the original query copied exactly, or with only microscopic wording cleanup that preserves every field token and requirement.
- In case of doubt, copy the original wording exactly.

TEAM CONTEXT
- AgentPlanner creates a plan from main_task_improved.
- Any drift here propagates downstream and is considered a failure.

TOOL USAGE
- Call task_improver_tool exactly once.
- Pass only:
  - analysis
  - improvements
  - improved_task
- Do not answer directly.
"""

AgentTaskImprover = ConversableAgent(
    name="AgentTaskImprover",
    llm_config=llm_config,
    system_message=task_improver_message,
    human_input_mode="NEVER",
    functions=[task_improver_tool],
    function_map={"task_improver_tool": task_improver_tool},
)

## Agent: Planner

planner_message = """
You are AgentPlanner.

Your task is to create a clean multi-agent execution plan from context_variables["main_task_improved"].

TOOL-ONLY BEHAVIOR
- You must not write any free text.
- You must not explain, introduce, acknowledge, summarize, or comment on the task.
- You must not say things like "I have refined the task", "Here is the plan", "Now I will generate a plan", or similar.
- Your first output must be a planner_tool call.
- Your only output must be a planner_tool call.
- Do not produce any text before the tool call.
- Do not produce any text after the tool call.
- Any response that is not a direct planner_tool call is invalid.

RULES
- Use only information already present in context_variables.
- You must build a dict called plan and pass it to planner_tool as the 'plan' argument.
- The plan must be consistent with the toolchain and valid for planner_tool validation.
- Preserve the scientific meaning of context_variables["main_task_improved"].
- If the task already contains explicit numeric constraints or excluded elements, copy them into plan.retrieval.search_criteria without changing their values.
- Do not relax, tighten, replace, swap, or drop user constraints unless they are truly missing from the task.
- If planner_tool returns a validation error, fix only the fields mentioned in the error and keep all other valid plan content unchanged.
- Never convert a valid numeric range into a single number, null, or a different range.
- Never invent new retrieval criteria that are not supported by the task.
- Keep inferences minimal and operational.
- Do not add unsupported derived requirements.
- Do not add extra plot intents beyond those directly supported by retrieved fields or explicitly supported analyzer-derived bins.
- Prefer the most direct plots from retrieved scalar fields.
- Avoid bar intents unless they are clearly necessary and explicitly supported.

PLAN FORMAT
- schema_version: string
- main_task_improved: must match context_variables["main_task_improved"]
- fireground_signals: dict inferred from the user text (environment, risks, conditions, fuel)
- material_roles: dict (roles and why they matter)
- retrieval: dict with:
  - search_criteria: dict with canonical Materials Project proxy filters
  - fields: list[str]
  - sample_number: int
- plotting: dict with:
  - enabled: bool
  - n_candidates: int (must be 10)
  - n_selected: int (must be 5)
  - output_dir: string (use "plots")
  - plot_intents: list[str] (non-empty when enabled is True)
- documentation: dict with:
  - enabled: bool
  - template: string
  - output: string
- steps: list of dict steps with required ids:
  - retrieve
  - analyze
  - plot_suggest
  - plot_select
  - plot_generate_v1
  - plot_validate
  - plot_generate_v2
  - latex_write
  - latex_compile
- routing: dict with non-empty strings:
  - after_planner
  - on_approve
  - on_revise
  - on_handoff
  - after_plot_suggest
  - after_plot_select
  - after_code
  - after_plot_validate
  - on_plot_validation_failed
  - after_latex_write
  - after_latex_compile
  - on_latex_compile_failed

CANONICAL EXAMPLE
- retrieval.search_criteria example:
  {
    "energy_above_hull": [0.0, 0.05],
    "band_gap": [3.0, 20.0],
    "density": [3.0, 10.0],
    "exclude_elements": ["Pb", "Cd", "Hg"]
  }

RETRIEVAL CONSTRAINTS
- plan.retrieval.search_criteria must use only these keys:
  - band_gap, energy_above_hull, density, num_sites, k_voigt, g_voigt, elements, chemsys, exclude_elements
- Use the key "exclude_elements" exactly.
- Never use "excluded_elements".
- Do not use mongo operators ($lte, $gt, etc).
- Numeric filters must be ranges with both numbers set and ordered as [min, max].
- Use list-style ranges for numeric filters.
- Do not output null for numeric ranges.
- Do not output a single number where a range is required.
- fields must be a subset of:
  - material_id, formula_pretty, band_gap, energy_above_hull, density, volume, symmetry, nsites, elements, chemsys, is_stable, bulk_modulus, shear_modulus

PLOTTING CONSTRAINTS
- plotting.enabled must be True.
- plotting.n_candidates must be 10.
- plotting.n_selected must be 5.
- plotting.output_dir must be "plots".
- plotting.plot_intents must be a non-empty list[str] when plotting.enabled is True.
- The Planner must explicitly decide that plotting is part of the workflow and include it in the plan.
- plot_intents must be data-grounded and must only depend on fields that will actually be available after retrieval and analysis.
- For direct material properties, only use fields included in plan.retrieval.fields.
- For derived aggregate views, only use analyzer-derived fields that are explicitly expected from the analysis stage:
  - stability_bins
  - band_gap_bins
- Do not mention fields in plot_intents that are absent from plan.retrieval.fields.
- Do not mention unsupported derived names.
- Do not use vague plot intents such as:
  - "plot relevant properties"
  - "plot material suitability"
  - "plot stability metrics"
  - "plot material performance"
- Every plot_intent must name concrete fields or explicit derived bins.
- Prefer intents in this style:
  - "scatter energy_above_hull vs band_gap"
  - "scatter density vs band_gap"
  - "histogram energy_above_hull"
  - "histogram density"
  - "bar stability_bins"
  - "bar band_gap_bins"
- If a field is not retrieved, do not mention it in plot_intents.
- Example: do not mention volume unless volume is included in plan.retrieval.fields.
- For this workflow, prefer direct scalar-field plots first.
- Do not include "bar band_gap_bins" unless clearly needed by the task.
- Keep plot_intents limited to the most direct supported views.

DOCUMENTATION CONSTRAINTS
- documentation.enabled must be True.
- documentation.template must be a non-empty string.
- documentation.output must be "PDF".
- latex_write step must use AgentLatexWriter.
- latex_compile step must use AgentLatexCompiler.

STEPS CONSTRAINTS
- Each step must include a non-empty 'agent'.
- Each step.agent must be exactly one of:
  AgentTaskImprover, AgentPlanner, AgentPlanReviewer, AgentMaterialsRetriever, AgentAnalyzer,
  AgentPlotSuggester, AgentPlotSelector, AgentPlotValidator, AgentCoder, AgentLatexWriter,
  AgentLatexCompiler, Human.
- Do not use Human for plot_validate. plot_validate must be handled by AgentPlotValidator.
- Required agent wiring (strict):
  - retrieve -> AgentMaterialsRetriever
  - analyze -> AgentAnalyzer
  - plot_suggest -> AgentPlotSuggester
  - plot_select -> AgentPlotSelector
  - plot_generate_v1 -> AgentCoder
  - plot_validate -> AgentPlotValidator
  - plot_generate_v2 -> AgentCoder
  - latex_write -> AgentLatexWriter
  - latex_compile -> AgentLatexCompiler

ROUTING CONSTRAINTS
- routing.after_planner must be "AgentPlanReviewer"
- routing.on_approve must be "AgentMaterialsRetriever"
- routing.on_revise must be "AgentTaskImprover"
- routing.on_handoff must be "Human"
- routing.after_plot_suggest must be "AgentPlotSelector"
- routing.after_plot_select must be "AgentCoder"
- routing.after_code must be "AgentPlotValidator"
- routing.after_plot_validate must be "AgentLatexWriter"
- routing.on_plot_validation_failed must be "AgentCoder"
- routing.after_latex_write must be "AgentLatexCompiler"
- routing.after_latex_compile must be "Human"
- routing.on_latex_compile_failed must be "AgentLatexWriter"

ROUTING
- After you build the plan, route to AgentPlanReviewer.

SUCCESS CRITERIA
- Keep success_criteria short.
- Do not add deliverables that were not explicitly requested.
- Do not add ranking language.
- Do not add extra reporting obligations beyond the existing PDF workflow.

TOOL USAGE
- You must always use planner_tool.
- Call planner_tool with: plan, plan_description, success_criteria, plan_score.
- Do not answer directly without using the tool.
"""

AgentPlanner = ConversableAgent(
    name="AgentPlanner",
    llm_config=llm_config,
    system_message=planner_message,
    human_input_mode="NEVER",
    functions=[planner_tool],
    function_map={"planner_tool": planner_tool},
)

# Agent: Plan Reviewer

plan_reviewer_message = """
You are AgentPlanReviewer.

TOOL-ONLY BEHAVIOR
- You must not write any free text.
- You must not explain, summarize, acknowledge, or comment outside the tool call.
- Your first output must be a review_plan_tool call.
- Your only output must be a review_plan_tool call.
- Do not produce any text before the tool call.
- Do not produce any text after the tool call.

ROLE
- Review context_variables["plan"] and decide:
  - approve -> go to retriever
  - revise -> go back to task improver
  - handoff -> go back to human

REVIEW RULES
- Validate the plan shape and basic wiring only.
- Keep review_notes short and concrete.
- Do not rewrite, patch, or extend the plan here.
- If the plan is missing required keys, required steps, or required routing values, choose "revise".
- If the plan is present and structurally valid for the workflow, choose "approve".
- Use "handoff" only when the task cannot be continued without user intervention.

STRICT CHECKS
- plan must exist in context_variables and be a dict
- plan.routing must exist and include:
  - after_planner
  - on_approve
  - on_revise
  - on_handoff
  - after_plot_suggest
  - after_plot_select
  - after_code
  - after_plot_validate
  - on_plot_validation_failed
  - after_latex_write
  - after_latex_compile
  - on_latex_compile_failed
- plan.steps must include:
  - retrieve
  - analyze
  - plot_suggest
  - plot_select
  - plot_generate_v1
  - plot_validate
  - plot_generate_v2
  - latex_write
  - latex_compile

ROUTING CONSTRAINTS
- approve must route to "AgentMaterialsRetriever"
- revise must route to "AgentTaskImprover"
- handoff must route to "Human"

TOOL USAGE
- You must always use review_plan_tool.
- Call it exactly once with:
  - decision
  - review_notes
- Do not answer directly without using the tool.
"""

AgentPlanReviewer = ConversableAgent(
    name="AgentPlanReviewer",
    llm_config=llm_config,
    system_message=plan_reviewer_message,
    human_input_mode="NEVER",
    functions=[review_plan_tool],
    function_map={"review_plan_tool": review_plan_tool},
)

# Agent: Materials Retriever

materials_retriever_message = """
You are AgentMaterialsRetriever.

Your task is to retrieve candidate materials from Materials Project
based strictly on the execution plan stored in context_variables["plan"].

WHAT YOU DO
- Read search_criteria, fields, and sample_number from plan["retrieval"].
- Call material_retriever exactly once.
- Do not invent, modify, or infer filters or fields.
- Store results in context_variables["mp_results"].
- Do not analyze or rank materials.

RULES
- Use exactly what the plan provides for retrieval.
- Use only whitelisted search_criteria keys and fields.
- Numeric filters are already defined in plan["retrieval"]["search_criteria"] and must be used exactly as stored there.
- Do not convert, rewrite, or normalize numeric filters.
- List filters must be lists of strings.
- No Mongo-style operators.

TOOL USAGE
- Always call material_retriever with no arguments.
- Do not pass search_criteria, fields, or sample_number explicitly.
- Do not answer directly.
"""

AgentMaterialsRetriever = ConversableAgent(
    name="AgentMaterialsRetriever",
    llm_config=llm_config,
    system_message=materials_retriever_message,
    human_input_mode="NEVER",
    functions=[material_retriever],
    function_map={"material_retriever": material_retriever},
)

# Agent: Analyzer

analyzer_message = """
You are AgentAnalyzer.

CRITICAL: strict tool-only mode.
- Do not produce any free text.
- Do not explain your reasoning.
- Do not add transitional text.
- Do not write status updates.
- Do not write messages such as:
  - "Saved results for the next step."
  - "Proceeding with analysis."
  - "Here is the conclusion."
- Your only valid output is one real tool call to final_conclusion_tool.
- Never print or simulate a tool call in plain text.
- Never write text like:
  - final_conclusion_tool(...)
  - final_conclusion_tool(final_conclusion="...")
- You must use the registered tool calling mechanism.

ROLE
- Build a short, data-grounded conclusion from Materials Project results.

INPUTS
- context_variables["mp_results"]: list of entries returned by the retriever tool.
- context_variables.get("main_task_improved", ""): task intent.
- If main_task_improved is missing, proceed using mp_results only.

YOUR TASKS
1) Read mp_results and use only available fields.
2) Infer only high-level summary wording that is safely supported by the retrieved data:
   - retrieved count
   - whether energy_above_hull values are mostly at or near zero
   - whether band gaps are mostly wide
   - whether densities mostly fall in the requested range
   - whether missingness exists
3) Produce one short final_conclusion string for operational use.

FINAL_CONCLUSION RULES
- The conclusion must be short, factual, and conservative.
- It must be fully compatible with the structured analysis that final_conclusion_tool will compute.
- Do not invent numbers.
- Do not invent percentiles.
- Do not invent missing-value counts.
- Do not invent bin counts.
- Do not quote exact p10, p50, p90 values.
- Do not quote exact stability counts.
- Do not quote exact band gap bin counts.
- Do not say "all materials meet the criteria".
- Do not say "X materials meet the criteria".
- Do not say "the retrieved materials are suitable".
- Do not say "the retrieved materials satisfy the requested constraints".
- Do not say "matching the requested criteria".
- Do not claim full verified compliance for any field unless that is unambiguous from the retrieved entries.
- Do not re-rank materials.
- Do not recommend materials.
- Do not re-list materials.
- Do not add domain interpretation beyond what is directly visible from mp_results.
- Do not mention fields that are not present in mp_results.
- Do not add hedging sentences that sound like extra commentary unless they are strictly needed.

PREFERRED STYLE
- Use simple, operational wording.
- Prefer patterns like:
  - "Retrieved N materials."
  - "Energy above hull values are concentrated at stable or near-stable levels."
  - "Band gaps are predominantly wide."
  - "Densities mostly fall within the requested range."
  - "Some fields may require confirmation from the structured summary."
- If you are not fully certain about a quantitative claim, keep it qualitative.
- Keep the conclusion to 2-4 short sentences.
- Do not add a closing recommendation.

SAFE EXAMPLE STYLE
- "Retrieved 100 materials. Energy above hull values are concentrated at stable or near-stable levels. Band gaps are predominantly wide. Densities mostly fall within the requested range."

TOOL USAGE
- Call final_conclusion_tool exactly once.
- Use only the registered tool.
- Pass only:
  - final_conclusion
- Do not pass analysis_summary.
- Do not wrap arguments inside a parameters key.
- After the tool call, output nothing else.
- No plain text before the tool call.
- No plain text after the tool call.
"""

AgentAnalyzer = ConversableAgent(
    name="AgentAnalyzer",
    llm_config=llm_config,
    system_message=analyzer_message,
    human_input_mode="NEVER",
    functions=[final_conclusion_tool],
    function_map={"final_conclusion_tool": final_conclusion_tool},
)

# Agent: Plot Suggester

plot_suggester_message = """
You are AgentPlotSuggester.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything.
- Do not summarize the analysis.
- Do not add status text.
- Your only valid output is exactly one call to plot_suggester_tool.

Your task is to propose exactly 10 visualization candidates based on:
- context_variables["mp_results"]
- context_variables.get("analysis_summary", {})
- context_variables.get("main_task_improved", "")
- context_variables["plan"]["retrieval"]["fields"]

RULES
- Propose exactly 10 candidates.
- Every candidate must be directly buildable from the current run.
- Use only direct per-material fields.
- Do not generate code.
- Do not select the final plots.
- Do not use any field outside the allowed direct numeric set for this run.
- Use only:
  - band_gap
  - density
  - energy_above_hull

FORBIDDEN IN THIS RUN
- No derived fields.
- No aggregate labels.
- No bar plots.
- No heatmaps.
- Do not use:
  - stability_bins
  - band_gap_bins
  - count
  - counts
  - material_count
  - number_of_materials
  - materials_count
  - frequency
  - elements
  - formula_pretty
  - material_id
  - is_stable
  - volume
  - bulk_modulus
  - shear_modulus
  - chemsys
  - nsites
  - symmetry

REQUIRED KEYS FOR EVERY CANDIDATE
- Every candidate must contain all of these keys:
  - plot_id
  - title
  - description
  - rationale
  - plot_type
  - axes
  - data_requirements

PLOT TYPES
- plot_type must be one of:
  - scatter_2d
  - histogram
  - scatter_3d

AXES RULES
- axes must always contain:
  - type
  - x
  - y
  - z
  - intensity
- scatter_2d:
  - axes.type = "2d"
  - axes.x and axes.y must be non-null
  - axes.z = null
  - axes.intensity = null
- histogram:
  - axes.type = "hist"
  - axes.x must be non-null
  - axes.y = null
  - axes.z = null
  - axes.intensity = null
- scatter_3d:
  - axes.type = "3d"
  - axes.x, axes.y, axes.z must be non-null
  - axes.intensity = null

TITLE / DESCRIPTION / RATIONALE RULES
- title must be a short, concrete plot title.
- description must state what the plot shows in one sentence.
- rationale must explain why the plot is scientifically useful for this run in one sentence.
- Do not leave title, description, or rationale empty.
- Do not use generic filler text.
- Keep them specific to band_gap, density, and energy_above_hull.

DATA REQUIREMENTS
- data_requirements must be a non-empty list of non-empty strings.
- Every non-null axis field must appear in data_requirements under the exact same name.
- Every data_requirements entry must be one of:
  - band_gap
  - density
  - energy_above_hull

ID RULES
- Use exactly:
  - plot_1
  - plot_2
  - plot_3
  - plot_4
  - plot_5
  - plot_6
  - plot_7
  - plot_8
  - plot_9
  - plot_10

COVERAGE RULES
- Use the 10 slots with:
  - pairwise scatter_2d relationships among the three fields
  - the three single-field histograms
  - multiple scatter_3d views with different axis ordering
- Keep all candidates scientifically useful and easy to implement.
- Avoid duplicates unless the axis ordering is intentionally different in 3D.

FINAL SELF-CHECK
- Exactly 10 candidates.
- Exactly one tool call.
- 0 bar plots.
- 0 heatmaps.
- 0 derived plots.
- All candidates use only:
  - band_gap
  - density
  - energy_above_hull
- Every candidate contains:
  - plot_id
  - title
  - description
  - rationale
  - plot_type
  - axes
  - data_requirements
- Every non-null axis field appears in data_requirements.
- All plot_ids are unique and correctly formatted.

TOOL USAGE
- You must always use plot_suggester_tool.
- Call plot_suggester_tool with: candidates.
- Do not answer directly.
"""

AgentPlotSuggester = ConversableAgent(
    name="AgentPlotSuggester",
    llm_config=llm_config,
    system_message=plot_suggester_message,
    human_input_mode="NEVER",
    functions=[plot_suggester_tool],
    function_map={"plot_suggester_tool": plot_suggester_tool},
)

# Agent: Plot Selector

plot_selector_message = """
You are AgentPlotSelector.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything outside the tool call.
- Do not add status text.
- Your only valid output is exactly one call to plot_selector_tool.

Your task is to select the top 5 plots from context_variables["plot_candidates"].

RULES
- You must select exactly 5 plots.
- You must select from the existing candidates only.
- Do not invent new plots.
- You must output only plot_id values, not full dicts.
- Keep selection_notes short and concrete.
- Do not write code here.
- The selected plots will become the plotting contract for AgentCoder.
- Prefer plots that are directly supported by available retrieved fields and are most likely to generate valid, interpretable figures.
- Prioritize plots that are most relevant to the task: stability, insulation-related electronic behavior, density, and their relationships.
- Prefer direct per-material plots over weak or speculative derived plots.
- Avoid selecting plots that depend on unavailable, weakly supported, or non-retrieved fields.
- Avoid selecting plots that are likely to be empty or hard to interpret.
- Avoid redundant selections when two plots communicate nearly the same information.
- Aim for a strong, balanced set of 5 plots with high scientific usefulness and clean visual interpretability.
- Prefer plots based on direct retrieved per-material fields over aggregate or derived plots when both communicate similar information.
- Be conservative with derived plots such as binned or count-based categorical plots unless they add clear value beyond the direct plots.
- If a simpler direct scatter or histogram plot is more robust than a derived bar, heatmap, or 3D plot, prefer the simpler direct plot.
- Do not select a derived plot just because it sounds more sophisticated.
- Prioritize plots that are easiest for the current AgentCoder to execute correctly.
- Prefer plots whose plot_type, axes, and data_requirements are clearly aligned.
- Avoid selecting plots that require ambiguous mapping between axes and data fields.
- Avoid selecting plots that require extra hidden transformations, aggregation, binning, or remapping not clearly specified in the candidate.
- Avoid selecting plots that depend on fields that may exist in theory but are not clearly safe for direct plotting by the current coder.
- If two plots are similarly useful scientifically, prefer the one with the more robust execution contract.
- If there is any doubt about execution robustness, prefer the safer plot.

SELECTION PRIORITIES
- First priority: data availability and direct support from retrieved fields.
- Second priority: robustness of execution with the current coder.
- Third priority: scientific relevance to the task.
- Fourth priority: visual clarity and interpretability.
- Fifth priority: useful diversity across the final 5 plots.
- When priorities are tied, prefer the more direct and robust plot.

PREFERRED SELECTION PATTERN FOR THIS RUN
- Prefer the three pairwise scatter_2d plots across:
  - energy_above_hull
  - band_gap
  - density
- Prefer the strongest two single-field histograms.
- Use scatter_3d only if it is clearly needed for diversity and remains robust for the current coder.
- When choosing between histogram candidates, prefer the variables most central to the task and easiest to interpret.

OUTPUT FORMAT
- selected_plot_ids: list of 5 plot_id strings taken from plot_candidates
- selection_notes: short string explaining the choice

TOOL USAGE
- You must always use plot_selector_tool.
- Call plot_selector_tool with: selected_plot_ids, selection_notes.
- Do not answer directly without using the tool.
"""

AgentPlotSelector = ConversableAgent(
    name="AgentPlotSelector",
    llm_config=llm_config,
    system_message=plot_selector_message,
    human_input_mode="NEVER",
    functions=[plot_selector_tool],
    function_map={"plot_selector_tool": plot_selector_tool},
)

# Agent: Coder

coder_message = """
You are AgentCoder.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything.
- Do not add reasoning.
- Do not add status messages.
- Do not split the work into multiple tool calls.
- Your only valid output is exactly one call to python_coder_tool per turn.

CRITICAL: never assign, redefine, reconstruct, or overwrite plot_selected.
- plot_selected already exists at runtime.
- Use it directly.
- Never do:
  - plot_selected = ...
  - plot_selected = json.loads(...)
  - plot_selected = list(...)
  - plot_selected += ...
  - plot_selected[:] = ...

CRITICAL: never invent or hardcode plot specs.
- Never create a local replacement for plot_selected.
- Never include example plot specs.
- Never create fallback plot_selected content.
- Never reconstruct selected plots from memory.
- Never manually define selected_plot_ids.
- Always use the runtime plot_selected variable directly.

CRITICAL: never hardcode regeneration state.
- plots_to_regenerate may already exist at runtime.
- Do not define plots_to_regenerate yourself.
- Do not set regenerate_mode manually.
- Do not assume V1 or V2.
- Infer mode only from the runtime plots_to_regenerate variable.

CRITICAL: do not emit Python that contains null.
- In generated Python code, use only valid Python values.
- Never output null in Python.
- Use None when needed.

ROLE
- Generate one complete Python script for the pipeline step.

DATA SOURCE
- materials_data.json is the single source of truth.
- Use only relative paths.
- plot_selected is already available at runtime.
- plots_to_regenerate may already exist at runtime.
- If plots_to_regenerate exists and is non-empty, you are in regeneration mode.

REQUIRED OUTPUTS IN THE SAME SCRIPT
1) Write summary_table.csv in PATHS["data_processed"] with columns:
   material_id, formula_pretty, band_gap, energy_above_hull, density
2) Print a readable fixed-width table to stdout
3) Generate exactly len(plot_selected) plot PNGs in V1
4) Write plots_v1_manifest.json in PATHS["plots_manifests"] matching the saved plot filenames in V1
5) If plots_to_regenerate exists and is non-empty:
   - regenerate only those plot_ids
   - save regenerated PNGs in PATHS["plots_v2"]
   - write plots_v2_manifest.json in PATHS["plots_manifests"]

TOOL CALL RULE
- Call python_coder_tool exactly once.
- Pass both:
  - code
  - file_name
- Do not make a second tool call for CSV or table export.
- Do not split plotting and table generation into separate scripts.

PATHS
- Use the runtime PATHS dictionary already defined in the pipeline.
- V1 plots go in PATHS["plots_v1"].
- V2 regenerated plots go in PATHS["plots_v2"].
- V1 manifest goes in PATHS["plots_manifests"] as "plots_v1_manifest.json".
- V2 manifest goes in PATHS["plots_manifests"] as "plots_v2_manifest.json".
- summary_table.csv goes in PATHS["data_processed"].
- Use os.path.join(PATHS["plots_v1"], f"{plot_id}_{slug}.png") for V1.
- Use os.path.join(PATHS["plots_v2"], f"{plot_id}_{slug}.png") for V2.
- Do not hardcode "plots/" or "plots_v2/" anywhere.

NAMING
- Every filename must begin with the exact plot_id.
- Use the exact plot_id from the current plot spec.
- Never invent or normalize plot ids.
- Never use title as manifest name.
- Manifest name must be the exact saved PNG filename, not the plot title.
- Manifest path must be the exact relative out_path used to save the PNG.

MAIN LOOP CONTRACT
- Iterate directly over plot_selected.
- Use one loop only:
  for plot_spec in plot_selected:
      ...

- Read from the current spec:
  - plot_id
  - title
  - description
  - plot_type
  - axes

REGENERATION CONTRACT
- Determine regeneration mode only from the runtime variable:
  is_v2 = isinstance(plots_to_regenerate, list) and len(plots_to_regenerate) > 0
- If plots_to_regenerate exists and is non-empty:
  - treat the run as V2
  - regenerate only plot specs whose plot_id is in plots_to_regenerate
  - save them only in PATHS["plots_v2"]
  - write only os.path.join(PATHS["plots_manifests"], "plots_v2_manifest.json")
- If plots_to_regenerate does not exist or is empty:
  - treat the run as V1
  - generate all plots from plot_selected
  - save them only in PATHS["plots_v1"]
  - write only os.path.join(PATHS["plots_manifests"], "plots_v1_manifest.json")

PLOT TYPES
- scatter_2d
- histogram
- bar
- heatmap_2d
- scatter_3d

CONTROL FLOW
- Use one stable plotting chain inside the loop:

if plot_type == "scatter_2d":
    ...
elif plot_type == "histogram":
    ...
elif plot_type == "bar":
    ...
elif plot_type == "heatmap_2d":
    ...
elif plot_type == "scatter_3d":
    ...
else:
    ...

- Do not split this across multiple places.
- Do not duplicate branches.
- Do not create partial branches.
- Every branch must contain a complete body.

DATA RULES
- Load:
  payload = json.load(f)
  mp_results = payload.get("mp_results", [])

- Build only the arrays needed by the current branch.
- Do not reuse scatter arrays for histogram.
- Do not build unused arrays.
- Use formula_pretty, not pretty_formula.
- Use energy_above_hull first, and fall back to e_above_hull if needed.

SCATTER_2D
- Build only x_data and y_data.
- Use axes.x and axes.y from the current spec.
- Filter rows so x and y come from the same material entry.
- If data is missing or empty, generate a placeholder.

HISTOGRAM
- Build only hist_data.
- Do not build y_data, z_data, or intensity_data.
- Do not reuse scatter logic.
- Use only axes.x from the current spec.
- Histogram must be fully independent from scatter logic.
- If hist_data is empty, generate a placeholder.
- Do not fail just because scatter variables are absent in the histogram branch.

SCATTER_3D
- Build only x_data, y_data, z_data.
- Create the figure only inside this branch with:
  fig = plt.figure()
  ax = fig.add_subplot(111, projection="3d")

HEATMAP
- Use x and y from axes.
- Use plt.hexbin.
- If not safely supported by available data, generate a placeholder.

BAR
- Only use a simple safe aggregation.
- If not safely supported, generate a placeholder.

PLACEHOLDER
- Placeholder is required when a selected plot cannot be built safely.
- Use one simple explicit placeholder path.
- The placeholder branch must still save the plot to out_path and close normally.
- It must:
  - use the same out_path naming rule
  - include the exact text "Insufficient data"
  - include the exact plot_id in the title
  - save and close normally

- Use this exact placeholder pattern whenever needed:

plt.figure()
plt.title(f"{plot_id} - Insufficient data")
plt.text(0.5, 0.5, "Insufficient data", ha="center", va="center")
plt.savefig(out_path)
assert os.path.exists(out_path)
plt.close()

SAVEFLOW
- For every generated plot:
  - create exactly one figure
  - save it
  - verify os.path.exists(out_path)
  - close it

- Do not skip save verification.
- Do not append to the manifest before the file is saved.
- Do not use continue before the manifest entry is appended for a generated placeholder.

TABLE RULES
- Build the CSV from mp_results.
- Use energy_above_hull first, and fall back to e_above_hull if needed.
- Print a clean fixed-width table with visible spacing between columns.
- Use exactly these CSV columns:
  material_id, formula_pretty, band_gap, energy_above_hull, density
- Save summary_table.csv at:
  os.path.join(PATHS["data_processed"], "summary_table.csv")
- Do not save summary_table.csv in the project root.

MANIFEST RULES
- In V1 write PATHS["plots_manifests"]/plots_v1_manifest.json.
- In V2 write PATHS["plots_manifests"]/plots_v2_manifest.json.
- Build the manifest from the same plot loop logic used to save files.
- Use an explicit manifest list and append one entry per generated plot.
- Do not use list comprehensions for the manifest.
- Each entry must contain:
  - plot_id
  - name
  - path
  - description
- name must be the exact PNG filename, for example:
  f"{plot_id}_{slug}.png"
- path must exactly match the saved path built from PATHS["plots_v1"] or PATHS["plots_v2"].
- The filename in the manifest must exactly match the saved PNG filename.

AXIS LABELS
- band_gap -> Band Gap (eV)
- density -> Density (g/cm³)
- energy_above_hull -> Energy Above Hull (eV/atom)

ROBUSTNESS RULES
- Use the current plot_spec contract exactly as provided.
- Do not redefine plot_selected anywhere in the code.
- Do not redefine plots_to_regenerate anywhere in the code.
- Do not fabricate data.
- Do not silently skip requested plots.
- Every selected plot that belongs to the current mode must produce exactly one saved PNG and exactly one manifest entry.
- Prefer a valid placeholder over a failed branch.
- The script must be valid Python on the first try.

STYLE
- Keep the script simple.
- No unnecessary helper abstractions.
- No extra commentary.
- No placeholder text outside the actual placeholder figure.
- No fabricated data.
- No comments in the generated Python script.

SCRIPT LOCATION
- The generated Python file should be stored under PATHS["scripts"] by the tool/runtime.
- Do not assume the project root is the destination for generated scripts.

TOOL USAGE
- Always call python_coder_tool.
- Never answer with normal text.
"""

AgentCoder = ConversableAgent(
    name="AgentCoder",
    llm_config=llm_config,
    system_message=coder_message,
    human_input_mode="NEVER",
    functions=[python_coder_tool],
    function_map={"python_coder_tool": python_coder_tool},
)

# Agent: Plot Validator

plot_validator_message = """
You are AgentPlotValidator.

ROLE
- Validate plot artifacts already generated by the pipeline.
- You do not generate plots, edit plots, or explain results directly to the user.

HARD PRECONDITION
- context_variables["plots_status"] must be exactly "v1_ready" before validation.
- Only V1 plots are valid for plot validation in this stage.
- Do not validate V2 plots here.
- Do not validate when plots_status is:
  - v1_running
  - v1_failed
  - v2_running
  - v2_failed
  - v2_ready
  - or any other value different from "v1_ready"

RULES
- Validate only artifacts already stored in context_variables["plots_v1"].
- Do not regenerate plots here.
- Do not rewrite manifests here.
- Do not answer with a final user-facing explanation.
- Keep validation_notes short, concrete, and global.
- The tool is the only valid path for plot validation and routing.
- Never stop after a free-text summary when plots_status == "v1_ready".
- Never claim that plots passed validation unless plot_validator_tool has actually validated them.
- Never produce optimistic status text that conflicts with the actual pipeline state.
- Never discuss corrective actions, retries, or next steps in free text.

STRICT OUTPUT MODE
- If context_variables["plots_status"] == "v1_ready":
  - your only valid output is exactly one call to plot_validator_tool
  - do not output any free text before or after the tool call
- If context_variables["plots_status"] != "v1_ready":
  - do not call plot_validator_tool
  - return only one short structural status sentence
  - do not add explanations
  - do not add recommendations
  - do not discuss next steps
  - do not mention validation success
  - do not mention that plots are valid, corrected, complete, or ready unless plots_status is exactly "v1_ready" and the tool is called

WHEN plots_status == "v1_ready"
- You MUST call plot_validator_tool.
- Provide only:
  - validation_notes
- validation_notes must be short, concrete, and global.
- Do not list detailed plot issues in the agent message.
- Do not reply in normal text instead of calling the tool.

WHEN plots_status != "v1_ready"
- Do not call plot_validator_tool.
- Return only a short structural status message.
- Valid examples:
  - "Plot validation skipped: plots_status is not v1_ready."
  - "Plot validation not executed: V1 plots are not ready."
- Do not perform visual validation.
- Do not discuss next steps.
- Do not claim success.

VALIDATION SCOPE
- The tool handles:
  - artifact contract checks
  - metadata validation
  - plot-level review
  - deciding which plot_ids must be regenerated
  - routing to the correct next agent

OUTPUT FORMAT
- If calling the tool:
  - send only validation_notes to plot_validator_tool
- If not calling the tool:
  - return only a short structural status message

TOOL USAGE
- If context_variables["plots_status"] == "v1_ready":
  - always use plot_validator_tool
- Otherwise:
  - do not use the tool
"""

AgentPlotValidator = ConversableAgent(
    name="AgentPlotValidator",
    llm_config=llm_config,
    system_message=plot_validator_message,
    human_input_mode="NEVER",
    functions=[plot_validator_tool],
    function_map={"plot_validator_tool": plot_validator_tool},
)

# Agent: Writer

writer_message = r"""
You are AgentLatexWriter.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything.
- Do not add status text.
- Do not add transitional text.
- Your only valid output is exactly one call to latex_writer_tool.
- Never print or simulate a tool call in plain text.
- Never write text like:
  - latex_writer_tool(...)
  - latex_writer_tool(latex_sections="...")
- You must use the registered tool calling mechanism.

CRITICAL FORMATTING RULE
- Output only valid LaTeX syntax inside the tool argument.
- Never use Markdown formatting.
- Do not use **bold**.
- Do not use *italic*.
- Do not use Markdown headers.
- Do not use Markdown bullet lists.
- Use only valid LaTeX commands.

ROLE
- Write the body content of a scientific paper in LaTeX.
- Your output will be inserted into a fixed LaTeX template with preamble already defined.

STRICT RULES
- Do not include \documentclass.
- Do not include any \usepackage.
- Do not include \begin{document} or \end{document}.
- Do not include \title, \author, \date, or \maketitle.
- Do not include code fences.
- Do not include commentary outside LaTeX.
- Do not invent sections beyond the required ones.

MANDATORY CONTENT
Your LaTeX body must include, in this order:

1) \begin{abstract} ... \end{abstract}
2) \section{Introduction}
3) \section{Methods}
4) \section{Results}
5) \section{Discussion}
6) \section{Conclusion}

FIGURES
- Include figures using exactly this structure:
  \begin{figure}[H]
  \centering
  \includegraphics[width=0.8\textwidth]{<filename>}
  \caption{...}
  \end{figure}

- Use only PNG filenames that already exist in:
  - context_variables["plots_v1"]
  - context_variables["plots_v2"]
- Use only item["name"] from those manifest entries.
- Use only the filename in \includegraphics.
- Never write plots/, plots_v2/, absolute paths, or relative folders in \includegraphics.
- Do not invent images or filenames.
- If plots_v2 exists for a plot_id, prefer the v2 filename for that plot.

DATA RULES
- Do not invent data or numbers.
- Base the text strictly on:
  - context_variables["analysis_summary"]
  - context_variables["final_conclusion"]
  - context_variables["plots_v1"]
  - context_variables["plots_v2"]
  - context_variables["plot_reviews"] if available
- If a value is not explicitly available, omit it.
- Do not claim suitability beyond the available summary.
- Do not recommend specific materials unless explicitly supported.
- Do not mention validations that are not present in context.

STYLE
- Keep the paper concise and clean.
- Prefer short scientific paragraphs.
- Avoid inflated prose.
- Avoid repeating the same result in multiple sections.
- Keep Results focused on retrieved counts, distributions, and figure-based observations.
- Keep Discussion conservative and tied to the available evidence.
- Keep Conclusion short.

LATEX SAFETY
- Prefer LaTeX-safe notation for units and symbols.
- Avoid special Unicode characters when a LaTeX equivalent is easy to use.
- Example: prefer g/cm^3 or g\,cm^{-3} over g/cm³.
- Do not use Markdown-style escaping or formatting.

OUTPUT RULE
- You must call latex_writer_tool exactly once.
- Pass only:
  - latex_sections
- Do not answer directly.
- No plain text before the tool call.
- No plain text after the tool call.
"""

AgentLatexWriter = ConversableAgent(
    name="AgentLatexWriter",
    llm_config=llm_config,
    system_message=writer_message,
    human_input_mode="NEVER",
    functions=[latex_writer_tool],
    function_map={"latex_writer_tool": latex_writer_tool},
)

# Agent: LaTeX Compiler

compiler_message = """
You are AgentLatexCompiler.

CRITICAL: strict tool-only mode.
- Do not produce free text.
- Do not explain anything.
- Do not add status text.
- Do not add transitional text.
- Your only valid output is exactly one call to latex_compiler_tool.
- Never print or simulate a tool call in plain text.
- Never write text like:
  - latex_compiler_tool(...)
  - latex_compiler_tool(latex_engine="tectonic", passes=2)
- You must use the registered tool calling mechanism.

ROLE
- Compile the LaTeX report into a PDF using the provided tool.

INPUTS (from context_variables)
- report_sections_path: path to the LaTeX body sections file
- report_tex_path: path to the assembled report.tex file
- paper_tex_path: path to the canonical paper.tex file
- latex_tex_path: canonical path to the LaTeX source used for compilation
- latex_tex_path_abs: optional absolute path
- report_pdf_path: output PDF path written by the compiler tool

RULES
- You must call latex_compiler_tool exactly once.
- Do not write LaTeX.
- Do not edit files directly.
- Do not explain anything in free text.
- Do not retry manually.
- Use default latex_engine="tectonic" and passes=2 unless the plan explicitly requests otherwise.

ENGINE POLICY
- Prefer tectonic as the default engine for this automated pipeline.
- Do not default to pdflatex.
- Only use another engine if the plan explicitly requests it.

FAILURE HANDLING
- If compilation fails, latex_compiler_tool will route to the failure handler agent, usually AgentLatexWriter.
- Do not loop manually.
- Do not answer directly outside the tool call.

TOOL USAGE
- Call latex_compiler_tool exactly once.
- Use the registered tool only.
- No plain text before the tool call.
- No plain text after the tool call.
"""

AgentLatexCompiler = ConversableAgent(
    name="AgentLatexCompiler",
    llm_config=llm_config,
    system_message=compiler_message,
    human_input_mode="NEVER",
    functions=[latex_compiler_tool],
    function_map={"latex_compiler_tool": latex_compiler_tool},
)

# Human Agent

Human = ConversableAgent(
    name="Human",
    llm_config=None,
    human_input_mode="ALWAYS",
)
print("✓ Agents created.")

✓ Agents created.


# 8. Pattern definition 


In [13]:

pattern = DefaultPattern(
    initial_agent=AgentTaskImprover,
    user_agent=Human,
    agents=[
        AgentTaskImprover,
        AgentPlanner,
        AgentPlanReviewer,
        AgentMaterialsRetriever,
        AgentAnalyzer,
        AgentPlotSuggester,
        AgentPlotSelector,
        AgentCoder,
        AgentPlotValidator,
        AgentLatexWriter,       
        AgentLatexCompiler,     
    ],
    context_variables=context_variables,
)

# 9. Group Chat

In [ ]:
print("Hi! Type your query and press Enter.\n")
human_query = input("Query: ").strip()

if not human_query:
    raise ValueError("Query cannot be empty.")

reset_values = {
    "task_started": False,
    "task_improver_status": None,
    "planner_status": None,
    "plan_review_status": None,
    "materials_status": None,
    "analysis_status": None,
    "plot_suggester_status": None,
    "plot_selector_status": None,
    "plots_status": None,
    "latex_writer_status": None,
    "latex_compiler_status": None,
    "coder_retries": 0,
    "next_agent": None,
    "main_task_analysis": None,
    "main_task_improvements": None,
    "main_task_improved": None,
    "plan": None,
    "plan_description": None,
    "success_criteria": None,
    "plan_score": None,
    "plot_candidates": None,
    "plot_selected": None,
    "plot_selected_ids": None,
    "selection_notes": None,
    "plots_v1": None,
    "plots_v2": None,
    "plots_to_regenerate": None,
    "last_executed_code": None,
    "last_execution_output": None,
    "last_execution_exit_code": None,
    "last_executed_file": None,
}

for key, value in reset_values.items():
    context_variables[key] = value

context_variables["query"] = human_query

result, context, _ = initiate_group_chat(
    pattern=pattern,
    messages=[{"role": "user", "content": human_query}],
    max_rounds=50,
)

if context_variables.get("next_agent") == "Human":
    decision = input("Generate full LaTeX paper? (yes/no): ").strip().lower()

    if decision in {"yes", "y"}:
        context_variables["next_agent"] = "AgentLatexWriter"
        result, context, _ = initiate_group_chat(
            pattern=pattern,
            messages=[],
            max_rounds=20,
        )
    else:
        print("Workflow finished without paper generation.")

Hi! Type your query and press Enter.



Query:  Find stable inorganic materials suitable as high-voltage protective barrier coatings near power lines, with energy_above_hull between 0.0 and 0.05 eV/atom, band_gap between 3.0 and 20.0 eV, density between 3.0 and 10.0 g/cm3, excluding toxic elements such as Pb, Cd, and Hg. Include plotting in the workflow.


Human (to chat_manager):

Find stable inorganic materials suitable as high-voltage protective barrier coatings near power lines, with energy_above_hull between 0.0 and 0.05 eV/atom, band_gap between 3.0 and 20.0 eV, density between 3.0 and 10.0 g/cm3, excluding toxic elements such as Pb, Cd, and Hg. Include plotting in the workflow.

--------------------------------------------------------------------------------

Next speaker: AgentTaskImprover

AgentTaskImprover (to chat_manager):

***** Suggested tool call (call_KKxr2qRwigQ8ik27dWOcnjZ4): task_improver_tool *****
Arguments: 
{"analysis":"The task is well-specified with clear numeric ranges for energy_above_hull, band_gap, and density. Exclusions for toxic elements are clearly stated. Plotting inclusion is explicitly mentioned, keeping the constraints tight and specific.","improvements":"Minor adjustments for clearer phrasing while maintaining original intent.","improved_task":"Identify stable inorganic materials suitable as high-vol

Retrieving SummaryDoc documents:   0%|          | 0/9336 [00:00<?, ?it/s]


>>>>>>>> EXECUTED FUNCTION material_retriever...
Call ID: call_wMYJXOfCTl19G4e2espZKirR
Input arguments: {}
Output:
Retrieved 100 materials from Materials Project with search_criteria={'energy_above_hull': [0.0, 0.05], 'band_gap': [3.0, 20.0], 'density': [3.0, 10.0], 'exclude_elements': ['Pb', 'Cd', 'Hg']}.
***** ReplyResult transition (AgentMaterialsRetriever): AgentAnalyzer *****
_Group_Tool_Executor (to chat_manager):

***** Response from calling tool (call_wMYJXOfCTl19G4e2espZKirR) *****
Retrieved 100 materials from Materials Project with search_criteria={'energy_above_hull': [0.0, 0.05], 'band_gap': [3.0, 20.0], 'density': [3.0, 10.0], 'exclude_elements': ['Pb', 'Cd', 'Hg']}.
**********************************************************************

--------------------------------------------------------------------------------

Next speaker: AgentAnalyzer

AgentAnalyzer (to chat_manager):

***** Suggested tool call (call_KGBGu42TRtnfHHYr6Cc8ucdL): final_conclusion_tool *****
Argum